# COVID-19 case study: severe cases *vs* controls (MetaboLights MTBLS2291)

This notebook recomputes everything in the article that concerns the COVID-19 cohort, from the deposited
feature table (`data/covid_0.5mz_1min.pkl`: 0.5 m/z × 1 min grid, 2–20 min, 21 600 bins). It covers
the stratified five-fold cross-validation of Table 5 and its ten-fold variants (Table S20), the
acquisition-batch analyses (batch-only baseline, mixed batches, leave-one-batch-out, coordinate-only
representations; Table S23), the PCA of Figure S6 and the downstream inspection of the most important
bins, the batch table S22 and the MS-level sensitivity of Table S21. Sections 6–10 read the 63 raw `_2` injections when they are available
(`MINUT_RAW/covid/*_2.mzXML`): the per-sample normalizer and the m/z 118.086 channel, the C₁₈H₃₇NO₂
isomers and the M+1 truncation (response letter), the normalization and grid sensitivity (Limitations),
the 212 literature severity markers and their identities (Table S24), the valine grid experiment and the
test of the second lung configuration on this cohort; otherwise they reuse the per-sample tables of the
last raw pass.

In [1]:
import json, os, platform, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, sklearn
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.stats import mannwhitneyu, rankdata
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd()))
import minut as mn
from minut import GRIDS, H, K, Na, NH4, H2O, D13C, MIXED_BATCHES, bin_index, batch_ratios, peak, read_scans, van_elteren, bin_maximum

warnings.filterwarnings("ignore")
GRID = GRIDS["covid"]; GRID_EXT = (100.0, 700.0, 0.5, 0.5, 20.5, 1.0); NB = int(np.prod(mn.grid_shape(GRID)))
OUT = mn.RESULTS / "covid"; FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True); INPUTS = Path.cwd() / "inputs"
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 70)
N_JOBS = 4
print(f"Python {platform.python_version()}, numpy {np.__version__}, scipy {scipy.__version__}, scikit-learn {sklearn.__version__}")
print("raw COVID files:", mn.raw_available("covid"), "| MAF:", mn.raw_available("maf"))

Python 3.11.9, numpy 1.26.4, scipy 1.16.3, scikit-learn 1.3.0
raw COVID files: True | MAF: True


## 1. Cohort and acquisition batches (Table S22)

63 subjects, one injection each (the data-dependent MS/MS run `_2`): 32 severe cases and 31 controls, the
mild group being excluded. The acquisition batch is the date prefix of the run name; there are seven
batches, three of which contain severe cases only.

In [2]:
X, y, subj = mn.load_covid(); batches = subj.sample_name.str[:8].to_numpy(); names = subj.sample_name.to_numpy(); sev = y == 1
I, M, R = mn.decoded_blocks(X, GRID)
comp = pd.crosstab(pd.Series(batches, name="batch"), pd.Series(np.where(sev, "severe", "control"), name="class")); comp["mixed"] = (comp.control > 0) & (comp.severe > 0)
display(comp); comp.to_csv(OUT / "covid_batch_composition.csv")
in_mixed = np.isin(batches, MIXED_BATCHES)
print(f"{len(y)} subjects: {int(sev.sum())} severe, {int((~sev).sum())} controls; {X.shape[1]} descriptors ({NB} bins); mixed batches {MIXED_BATCHES}: {int(in_mixed.sum())} subjects ({int((~sev & in_mixed).sum())} controls, {int((sev & in_mixed).sum())} severe); severe cases in batches without controls: {int((sev & ~in_mixed).sum())}")

class,control,severe,mixed
batch,,,
13092020,8,1,True
15092020,11,4,True
20092020,4,5,True
22092020,0,6,False
23092020,0,10,False
24092020,0,1,False
26092020,8,5,True


63 subjects: 32 severe, 31 controls; 64800 descriptors (21600 bins); mixed batches ['13092020', '15092020', '20092020', '26092020']: 46 subjects (31 controls, 15 severe); severe cases in batches without controls: 17


## 2. Cross-validation (Table 5; Table S20)

Stratified five-fold cross-validation (shuffled, seed 2024) with the hybrid selector refitted in each
training fold: Gini pre-ranking of the 1 000 most important descriptors (1 000 trees, seed 0), then the
mean of the min-max scaled Mann–Whitney U (controls first) and Fisher scores, 100 descriptors kept, and a
100-tree forest (seed 0). We then repeat with ten folds and several fold assignments.

In [3]:
def covid_pipeline():
    return Pipeline([("select", mn.HybridSelector(100, n_jobs=N_JOBS)), ("forest", RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=N_JOBS))])

protocols = [("5-fold stratified, shuffled, seed 2024 (paper)", StratifiedKFold(5, shuffle=True, random_state=2024)), ("10-fold stratified, shuffled, seed 2024", StratifiedKFold(10, shuffle=True, random_state=2024)),
             ("10-fold stratified, unshuffled", StratifiedKFold(10, shuffle=False))] + [(f"10-fold stratified, shuffled, seed {s}", StratifiedKFold(10, shuffle=True, random_state=s)) for s in range(4)]
summ, folds = [], []
for label, cvs in protocols:
    t0 = time.time(); t, _, _ = mn.evaluate_splits(covid_pipeline(), X, y, list(cvs.split(X, y))); t.insert(0, "protocol", label); folds.append(t)
    s = mn.summarize_folds(t, label); s["elapsed_s"] = round(time.time() - t0); summ.append(s); print(f"{label:48s} accuracy {s['accuracy']:.4f} ± {s['accuracy_sd']:.4f}  AUC {s['auc']:.4f} ± {s['auc_sd']:.4f}  ({s['elapsed_s']} s)")
cv_table = pd.DataFrame(summ); cv_table.to_csv(OUT / "covid_cv10_summary.csv", index=False, float_format="%.6f"); pd.concat(folds).to_csv(OUT / "covid_cv10_folds.csv", index=False, float_format="%.6f")
paper = cv_table.iloc[0]; ten = cv_table.iloc[1]; others = cv_table.iloc[2:]
pd.DataFrame([dict(cv_accuracy=paper.accuracy, cv_sd=paper.accuracy_sd, cv_auc=paper.auc, auc_sd=paper.auc_sd, fold_auc=paper.fold_auc)]).to_csv(OUT / "covid.csv", index=False, float_format="%.6f")

5-fold stratified, shuffled, seed 2024 (paper)   accuracy 0.8538 ± 0.0998  AUC 0.8663 ± 0.1289  (27 s)


10-fold stratified, shuffled, seed 2024          accuracy 0.7881 ± 0.1849  AUC 0.8778 ± 0.1280  (61 s)


10-fold stratified, unshuffled                   accuracy 0.7571 ± 0.2437  AUC 0.8472 ± 0.1897  (60 s)


10-fold stratified, shuffled, seed 0             accuracy 0.8452 ± 0.1533  AUC 0.8972 ± 0.1542  (62 s)


10-fold stratified, shuffled, seed 1             accuracy 0.8143 ± 0.1886  AUC 0.9361 ± 0.1255  (60 s)


10-fold stratified, shuffled, seed 2             accuracy 0.8357 ± 0.1827  AUC 0.9333 ± 0.1423  (41 s)


10-fold stratified, shuffled, seed 3             accuracy 0.8095 ± 0.1733  AUC 0.9083 ± 0.0861  (38 s)


## 3. Acquisition batches (Table S23)

A batch-only baseline (one-hot batch indicators, same folds), the pipeline restricted to the four mixed
batches, leave-one-batch-out over the seven batches with the out-of-fold AUC pooled over the 46 subjects
of the mixed batches, and the representations without the intensity block (m/z + RT, m/z only).

In [4]:
def batch_only_baseline(batches_, y_, splits):
    Xl = pd.get_dummies(pd.Series(batches_)).to_numpy(dtype=float); rows, oof = [], np.full(len(y_), np.nan)
    for k, (tr, te) in enumerate(splits):
        f = RandomForestClassifier(n_estimators=100, random_state=0).fit(Xl[tr], y_[tr]); pr = f.predict_proba(Xl[te]); pr = pr[:, 1] if pr.shape[1] == 2 else np.full(len(te), float(f.classes_[0])); oof[te] = pr
        rows.append(dict(fold=k, n_test=len(te), accuracy=accuracy_score(y_[te], (pr >= 0.5).astype(int)), auc=roc_auc_score(y_[te], pr) if len(np.unique(y_[te])) == 2 else np.nan, held_out_group=";".join(sorted(set(map(str, batches_[te]))))))
    return pd.DataFrame(rows), oof

summ, folds = [], []
def record(t, label, extra=None):
    t.insert(0, "protocol", label); folds.append(t); s = mn.summarize_folds(t, label); s.update(extra or {}); summ.append(s); return s
cv5 = list(StratifiedKFold(5, shuffle=True, random_state=2024).split(X, y)); t, _ = batch_only_baseline(batches, y, cv5); s_batch = record(t, "batch-only baseline, 5-fold stratified (seed 2024), all 63")
Xm, ym, lm = X[in_mixed], y[in_mixed], batches[in_mixed]; cvm = list(StratifiedKFold(5, shuffle=True, random_state=2024).split(Xm, ym))
t, _, _ = mn.evaluate_splits(covid_pipeline(), Xm, ym, cvm, groups=lm); s_mix = record(t, "MINUT, mixed batches only (n=46), 5-fold stratified (seed 2024)")
tb, _ = batch_only_baseline(lm, ym, cvm); s_mixlot = record(tb, "batch-only baseline, mixed batches only (n=46), 5-fold")
logo = list(LeaveOneGroupOut().split(X, y, groups=batches)); t, oofp, oofd = mn.evaluate_splits(covid_pipeline(), X, y, logo, groups=batches)
s_logo = record(t, "MINUT, leave-one-batch-out (7 batches)", dict(pooled_oof_accuracy_all63=accuracy_score(y, oofd), pooled_oof_accuracy_mixed46=accuracy_score(y[in_mixed], oofd[in_mixed]), pooled_oof_auc_mixed46=roc_auc_score(y[in_mixed], oofp[in_mixed]), pooled_oof_auc_all63=roc_auc_score(y, oofp), severe_only_batches_accuracy=accuracy_score(y[~in_mixed], oofd[~in_mixed])))
tb, oofb = batch_only_baseline(batches, y, logo); record(tb, "batch-only baseline, leave-one-batch-out", dict(pooled_oof_accuracy_all63=accuracy_score(y, (oofb >= 0.5).astype(int))))
display(t[["held_out_group", "n_test", "n_test_control", "n_test_severe", "accuracy", "auc"]].round(3))
print(f"batch-only baseline: {s_batch['accuracy']:.3f} ± {s_batch['accuracy_sd']:.3f}, AUC {s_batch['auc']:.3f}; mixed batches: {s_mix['accuracy']:.3f} ± {s_mix['accuracy_sd']:.3f} / AUC {s_mix['auc']:.3f} ± {s_mix['auc_sd']:.3f} (batch alone {s_mixlot['accuracy']:.3f} / {s_mixlot['auc']:.3f}); "
      f"leave-one-batch-out pooled AUC over the 46 mixed-batch subjects {s_logo['pooled_oof_auc_mixed46']:.3f}")
variants = {"m/z + RT blocks (no intensity)": X[:, NB:], "m/z block only": X[:, NB:2 * NB]}; s_var = {}
for vname, Xv in variants.items():
    t, _, _ = mn.evaluate_splits(covid_pipeline(), Xv[in_mixed], ym, cvm, groups=lm); record(t, f"MINUT {vname}, mixed batches only (n=46), 5-fold stratified (seed 2024)")
    t, op, od = mn.evaluate_splits(covid_pipeline(), Xv, y, logo, groups=batches)
    s_var[vname] = record(t, f"MINUT {vname}, leave-one-batch-out (7 batches)", dict(pooled_oof_accuracy_all63=accuracy_score(y, od), pooled_oof_accuracy_mixed46=accuracy_score(y[in_mixed], od[in_mixed]), pooled_oof_auc_mixed46=roc_auc_score(y[in_mixed], op[in_mixed]), pooled_oof_auc_all63=roc_auc_score(y, op), severe_only_batches_accuracy=accuracy_score(y[~in_mixed], od[~in_mixed])))
    t, _, _ = mn.evaluate_splits(covid_pipeline(), Xv, y, list(StratifiedKFold(5, shuffle=True, random_state=2024).split(Xv, y))); record(t, f"MINUT {vname}, all 63, 5-fold stratified (seed 2024)")
    print(f"{vname}: leave-one-batch-out pooled AUC (mixed 46) {s_var[vname]['pooled_oof_auc_mixed46']:.3f}")
batch = pd.DataFrame(summ); batch.to_csv(OUT / "covid_batch_aware_summary.csv", index=False, float_format="%.6f"); pd.concat(folds).to_csv(OUT / "covid_batch_aware_folds.csv", index=False, float_format="%.6f")
pd.DataFrame(dict(sample=names, subject=subj.subject_id, batch=batches, y=y, in_mixed_batch=in_mixed, logo_oof_proba=oofp, logo_oof_pred=oofd)).to_csv(OUT / "covid_batch_aware_logo_predictions.csv", index=False)

,held_out_group,n_test,n_test_control,n_test_severe,accuracy,auc
0,13092020,9,8,1,0.778,0.812
1,15092020,15,11,4,0.267,0.648
2,20092020,9,4,5,0.556,0.100
3,22092020,6,0,6,0.833,NaN
4,23092020,10,0,10,0.900,NaN
5,24092020,1,0,1,1.000,NaN
6,26092020,13,8,5,0.231,0.325


batch-only baseline: 0.742 ± 0.103, AUC 0.815; mixed batches: 0.867 ± 0.083 / AUC 0.922 ± 0.069 (batch alone 0.671 / 0.632); leave-one-batch-out pooled AUC over the 46 mixed-batch subjects 0.537


m/z + RT blocks (no intensity): leave-one-batch-out pooled AUC (mixed 46) 0.734


m/z block only: leave-one-batch-out pooled AUC (mixed 46) 0.735


## 4. PCA of the feature vectors (Figure S6)

Intensity block, standardized per descriptor. The first components follow the batches, not the classes.

In [5]:
Z = StandardScaler().fit_transform(I); pcs = PCA(n_components=5, random_state=0).fit(Z); P = pcs.transform(Z); ev = pcs.explained_variance_ratio_ * 100
order = sorted(set(batches), key=lambda s: pd.to_datetime(s, format="%d%m%Y"))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.6)); cmap = plt.cm.tab10(np.linspace(0, 1, len(order)))
for c, l in zip(cmap, order):
    m_ = batches == l; so = (m_ & ~sev).sum() == 0
    a1.scatter(P[m_, 0], P[m_, 1], s=42, color=c, edgecolor="k", linewidth=0.9 if so else 0.3, marker="D" if so else "o", label=pd.to_datetime(l, format="%d%m%Y").strftime("%d %b") + (" (severe only)" if so else ""))
a1.set_title("colored by acquisition batch", fontsize=10); a1.legend(fontsize=7, loc="best", framealpha=0.9)
for cat, col, mk, lab in ((~sev, "#ff7f0e", "o", "control"), (sev, "#1f77b4", "s", "severe")):
    a2.scatter(P[cat, 0], P[cat, 1], s=42, color=col, edgecolor="k", linewidth=0.3, marker=mk, label=lab)
a2.scatter(P[~in_mixed, 0], P[~in_mixed, 1], s=120, facecolors="none", edgecolors="k", linewidth=1.1, marker="D", label="severe-only batches"); a2.set_title("colored by category", fontsize=10); a2.legend(fontsize=8, loc="best", framealpha=0.9)
for ax in (a1, a2):
    ax.set_xlabel(f"PC1 ({ev[0]:.0f}%)"); ax.set_ylabel(f"PC2 ({ev[1]:.0f}%)"); ax.tick_params(labelsize=8)
fig.suptitle("PCA of the COVID-19 MINUT feature vectors (intensity block, 63 subjects)", fontsize=11); plt.tight_layout(); fig.savefig(FIG / "figS6_covid_pca.pdf"); fig.savefig(FIG / "figS6_covid_pca.png", dpi=160); plt.close(fig)
pd.DataFrame(dict(sample=names, batch=batches, severe=sev.astype(int), PC1=P[:, 0], PC2=P[:, 1], PC3=P[:, 2])).to_csv(OUT / "covid_pca_scores.csv", index=False)
print("explained variance PC1-5 (%):", [round(v, 1) for v in ev])

explained variance PC1-5 (%): [17.8, 10.2, 5.9, 3.9, 3.5]


Downstream inspection of the ten most important bins (importance = maximum over the three descriptors of
the bin in the 1 000-tree pre-ranking forest of the selector, seed 0, fitted on the 63 subjects; descriptive
only), plus the two bins discussed in the text, 16044 and 2436: occupancy, dispersion of the retained maxima,
dominant coordinate group (full span 5 ppm × 15 s), DBSCAN m/z sub-clusters and flag.

In [6]:
imp1000 = RandomForestClassifier(n_estimators=1000, random_state=0, n_jobs=N_JOBS).fit(X, y).feature_importances_
per_bin = imp1000.reshape(3, NB).max(axis=0); order = np.argsort(per_bin)[::-1][:10]
rows = []
for b in list(order) + [16044, 2436]:
    r = mn.inspect_bin(mn.bin_coordinates(X, int(b), GRID), y, int(b), GRID, ("control", "severe"), 5e-6, 15.0)
    r.update(importance=float(per_bin[b]), rank=int((per_bin > per_bin[b]).sum()) + 1); rows.append(r)
insp = pd.DataFrame(rows); insp.to_csv(OUT / "downstream_inspection_covid.csv", index=False, float_format="%.6g")
display(insp[["rank", "bin", "bin_limits", "occupancy", "mz_sd_ppm", "rt_sd_min", "support_D", "consistency_C", "n_mz_subclusters", "class_counts_present", "flag"]].round(3))
top10 = insp.iloc[:10]

,rank,bin,bin_limits,occupancy,mz_sd_ppm,rt_sd_min,support_D,consistency_C,n_mz_subclusters,class_counts_present,flag
0,1,20914,"m/z [357,357.5) RT [19,20) min",1.0,112.360,0.417,0.571,0.571,2,31 control/32 severe,mixed column: sub-cluster before any chemical reading
1,2,20177,"m/z [588.5,589) RT [18,19) min",1.0,90.863,0.226,0.524,0.524,1,31 control/32 severe,dispersed coordinates: inspect per-sample points
2,3,17009,"m/z [204.5,205) RT [16,17) min",1.0,59.777,0.222,0.698,0.698,2,31 control/32 severe,mixed column: sub-cluster before any chemical reading
3,4,18402,"m/z [301,301.5) RT [17,18) min",1.0,1.125,0.203,0.587,0.587,1,31 control/32 severe,dispersed coordinates: inspect per-sample points
4,5,16108,"m/z [354,354.5) RT [15,16) min",1.0,9.578,0.030,0.667,0.667,1,31 control/32 severe,dispersed coordinates: inspect per-sample points
5,6,15717,"m/z [158.5,159) RT [15,16) min",1.0,0.996,0.119,0.714,0.714,1,31 control/32 severe,dispersed coordinates: inspect per-sample points
6,7,5043,"m/z [221.5,222) RT [6,7) min",1.0,644.316,0.185,0.683,0.683,2,31 control/32 severe,mixed column: sub-cluster before any chemical reading
7,8,188,"m/z [194,194.5) RT [2,3) min",1.0,93.021,0.268,0.460,0.460,2,31 control/32 severe,mixed column: sub-cluster before any chemical reading
8,9,10922,"m/z [161,161.5) RT [11,12) min",1.0,175.919,0.265,0.476,0.476,2,31 control/32 severe,mixed column: sub-cluster before any chemical reading
9,10,13693,"m/z [346.5,347) RT [13,14) min",1.0,188.804,0.331,0.333,0.333,1,31 control/32 severe,dispersed coordinates: inspect per-sample points


## 5. Which spectra enter the classifier: MS-level sensitivity (Table S21)

The deposited `_2` vectors were binned from every scan of the file, profile MS$^1$ and centroided MS$^2$
together. Row 2 recomputes the vectors of the same injections from the vendor-centroided MS$^1$ scans with
`transform_file` (raw files needed). The full-MS injections `_3` and `_4` are not part of the local raw
set, so their rows come from the analytical matrix of the first revision, `data/covid_mz0p5_rt1.npz`: the 303
MTBLS2291 injections (63 patients × `_2`/`_3`/`_4`, QC and blanks) binned on the same 0.5 m/z × 1 min grid from the
vendor-centroided MS$^1$ scans (float32 coordinates, presence mask). It is stored with Git LFS like the feature
tables (`git lfs pull`); another location can be given in `MINUT_REEXTRACTED`. Without the file the cell prints a
note and skips those rows.

In [7]:
def layout_from_npz(z):
    mz_min, mz_max, _, rt_min, rt_max, _ = GRID; Ii = z["intensity_raw"].astype(np.float64); present = z["present"]
    mz = np.where(present, z["mz"].astype(np.float64), mz_min); rt = np.where(present, z["rt"].astype(np.float64), rt_min); Ii = np.where(present, Ii, 0.0); scale = Ii.max(axis=1, keepdims=True); scale[scale == 0] = 1
    return np.concatenate([Ii / scale, (mz - mz_min) / (mz_max - mz_min), (rt - rt_min) / (rt_max - rt_min)], axis=1)

def cv_rows(label, Xv, yv, ks=(5,)):
    out = []
    for k in ks:
        t0 = time.time(); t, _, _ = mn.evaluate_splits(covid_pipeline(), Xv, yv, list(StratifiedKFold(k, shuffle=True, random_state=2024).split(Xv, yv))); nm = f"{label}, {k}-fold stratified (seed 2024)"
        s = mn.summarize_folds(t, nm); s["n"] = len(yv); s["elapsed_s"] = round(time.time() - t0); out.append(s); print(f"{nm:62s} accuracy {s['accuracy']:.4f} ± {s['accuracy_sd']:.4f}  AUC {s['auc']:.4f} ± {s['auc_sd']:.4f}")
    return out

ms_rows = cv_rows("deposited _2 (profile MS1 + MS2)", X, y)
ms1_file = OUT / "covid_ms1only_2_vectors.npy"
if not ms1_file.exists() and mn.raw_available("covid"):
    t0 = time.time(); V2 = np.array(Parallel(n_jobs=6)(delayed(mn.transform_file)(next((mn.RAW / "covid").glob(f"{n}.mzXML")), GRID, (1,)) for n in names)); np.save(ms1_file, V2); print(f"MS1-only vectors of the 63 _2 files: {time.time() - t0:.0f} s")
if ms1_file.exists():
    V2 = np.load(ms1_file); same_pt = []
    for i in range(len(names)):
        dep_i, re_i = X[i, :NB], V2[i, :NB]; both = (dep_i > 0) & (re_i > 0)
        same_pt.append(dict(sample=names[i], bins_nonzero_deposited=int((dep_i > 0).sum()), bins_nonzero_ms1only=int((re_i > 0).sum()), bins_same_winning_point=int((both & np.isclose(M[i], 100 + 600 * V2[i, NB:2 * NB], atol=5e-4, rtol=0) & np.isclose(R[i], 2 + 18 * V2[i, 2 * NB:], atol=1e-3, rtol=0)).sum())))
    agree = pd.DataFrame(same_pt); agree["frac_same_retained_of_deposited"] = agree.bins_same_winning_point / agree.bins_nonzero_deposited; agree.to_csv(OUT / "covid_ms_level_bin_agreement.csv", index=False, float_format="%.6g")
    print(f"deposited vs MS1-only (same files): median {agree.bins_nonzero_deposited.median():.0f} vs {agree.bins_nonzero_ms1only.median():.0f} non-empty bins; identical retained point in {100 * agree.frac_same_retained_of_deposited.median():.1f} % of the deposited non-empty bins (median)")
    ms_rows += cv_rows("centroided MS1-only _2 (same injections, transform_file)", V2, y)
npz = Path(os.environ.get("MINUT_REEXTRACTED", mn.DATA / "covid_mz0p5_rt1.npz"))   # analytical matrix of the first revision (data/, Git LFS)
if npz.exists() and npz.stat().st_size > 10_000:   # a Git LFS pointer that has not been pulled is a few hundred bytes
    z = np.load(npz, allow_pickle=True); re_names = pd.Series(z["sample_name"]); Xall = layout_from_npz(z)
    idx2 = np.array([re_names[re_names == n].index[0] for n in names]); ms_rows += cv_rows("centroided MS1-only _2 (same injections, analytical matrix)", Xall[idx2], y)
    for suf in ("3", "4"):
        want = [n.rsplit("_", 1)[0] + f"_{suf}" for n in names]; found = [w for w in want if (re_names == w).any()]; kp = np.array([w in found for w in want])
        ms_rows += cv_rows(f"centroided MS1-only _{suf} (full-MS injection)", Xall[np.array([re_names[re_names == w].index[0] for w in found])], y[kp], ks=(5, 10))
else:
    print(f"analytical matrix not available at {npz} (run 'git lfs pull' or set MINUT_REEXTRACTED): rows _3 / _4 of Table S21 skipped")
ms_level = pd.DataFrame(ms_rows); ms_level.to_csv(OUT / "covid_ms_level_summary.csv", index=False, float_format="%.6f"); display(ms_level[["protocol", "n", "accuracy", "accuracy_sd", "auc", "auc_sd"]].round(3))


deposited _2 (profile MS1 + MS2), 5-fold stratified (seed 2024) accuracy 0.8538 ± 0.0998  AUC 0.8663 ± 0.1289


MS1-only vectors of the 63 _2 files: 1222 s
deposited vs MS1-only (same files): median 21310 vs 21293 non-empty bins; identical retained point in 77.6 % of the deposited non-empty bins (median)


centroided MS1-only _2 (same injections, transform_file), 5-fold stratified (seed 2024) accuracy 0.8205 ± 0.1640  AUC 0.8468 ± 0.1295


centroided MS1-only _2 (same injections, analytical matrix), 5-fold stratified (seed 2024) accuracy 0.8038 ± 0.1661  AUC 0.8468 ± 0.1295


centroided MS1-only _3 (full-MS injection), 5-fold stratified (seed 2024) accuracy 0.7923 ± 0.0659  AUC 0.9095 ± 0.0652


centroided MS1-only _3 (full-MS injection), 10-fold stratified (seed 2024) accuracy 0.7929 ± 0.1572  AUC 0.9167 ± 0.1032


centroided MS1-only _4 (full-MS injection), 5-fold stratified (seed 2024) accuracy 0.7756 ± 0.0829  AUC 0.8948 ± 0.0590


centroided MS1-only _4 (full-MS injection), 10-fold stratified (seed 2024) accuracy 0.7738 ± 0.1318  AUC 0.9139 ± 0.0828


,protocol,n,accuracy,accuracy_sd,auc,auc_sd
0,"deposited _2 (profile MS1 + MS2), 5-fold stratified (seed 2024)",63,0.854,0.100,0.866,0.129
1,"centroided MS1-only _2 (same injections, transform_file), 5-fold s...",63,0.821,0.164,0.847,0.129
2,"centroided MS1-only _2 (same injections, analytical matrix), 5-fol...",63,0.804,0.166,0.847,0.129
3,"centroided MS1-only _3 (full-MS injection), 5-fold stratified (see...",63,0.792,0.066,0.910,0.065
4,"centroided MS1-only _3 (full-MS injection), 10-fold stratified (se...",63,0.793,0.157,0.917,0.103
5,"centroided MS1-only _4 (full-MS injection), 5-fold stratified (see...",63,0.776,0.083,0.895,0.059
6,"centroided MS1-only _4 (full-MS injection), 10-fold stratified (se...",63,0.774,0.132,0.914,0.083


## 6. Raw data: the per-sample normalizer and the m/z 118.086 channel

The intensity block of the deposited vectors is divided by the per-sample maximum of each run. One pass over
the 63 `_2` files (centroided MS$^1$) records that maximum and its ion, the total ion current, the true
C₅H₁₂NO₂⁺ peak at 0.82 min and the 118.086 level inside the [3,4) and [4,5)-min windows of the grid. For
each quantity we give the severe/control ratio within each mixed batch (van Elteren test).

In [8]:
VAL = 118.08626
def files_covid():
    return sorted(str(p) for p in (mn.RAW / "covid").glob("*_2.mzXML"))
def disease(name):
    return "Severe" if "Severe" in name else "Control"

def process_118(path):
    name = Path(path).stem; rows = []; gmax = 0.0; gmz = gmrt = np.nan; tic = 0.0
    for rt, mz, it in read_scans(path):
        if it.size == 0:
            continue
        tic += float(it.sum())
        if 2.0 <= rt < 20.0:
            w = (mz >= 100) & (mz < 700)
            if w.any():
                j = np.flatnonzero(w)[np.argmax(it[w])]
                if it[j] > gmax:
                    gmax, gmz, gmrt = float(it[j]), float(mz[j]), rt
        if rt <= 5.5:
            rows.append((rt, peak(mz, it, VAL), peak(mz, it, VAL + D13C)))
    t = pd.DataFrame(rows, columns=["rt", "i", "i1"]); main = t[(t.rt >= 0.5) & (t.rt < 2.0)]; b3, b4 = t[(t.rt >= 3) & (t.rt < 4)], t[(t.rt >= 4) & (t.rt < 5)]; top = main[main.i >= 0.5 * main.i.max()]
    return dict(sample=name, disease=disease(name), batch=name[:8], tic=tic, grid_max_I=gmax, grid_max_mz=round(gmz, 4), grid_max_rt=round(gmrt, 2), main_rt=round(main.rt[main.i.idxmax()], 3), main_apex=main.i.max(), main_M1_over_M=top.i1.sum() / top.i.sum(),
                bin3_max=b3.i.max(), bin3_min=b3.i.min(), bin3_nonzero=(b3.i > 0).mean(), bin4_max=b4.i.max(), bin4_min=b4.i.min(), bin4_nonzero=(b4.i > 0).mean(), first_scan_I=t.i.iloc[0])

per118 = OUT / "covid_mz118_channel_per_sample.csv"; d118 = mn.cached(per118)
if d118 is None and mn.raw_available("covid"):
    t0 = time.time(); d118 = pd.DataFrame(Parallel(n_jobs=6)(delayed(process_118)(f) for f in files_covid())).sort_values("sample"); d118.to_csv(per118, index=False); d118 = mn.cached(per118); print(f"{time.time() - t0:.0f} s")
HAVE_COVID_RAW = d118 is not None
if HAVE_COVID_RAW:
    sv = (d118.disease == "Severe").to_numpy(); lt = d118.batch.to_numpy()
    def qrow(label, v):
        return dict(quantity=label, median_ctl=np.median(v[~sv]), median_sev=np.median(v[sv]), batches_sev_over_ctl=[round(x, 3) for x in batch_ratios(v, sv, lt, MIXED_BATCHES)], vanElteren_p=van_elteren(v, sv, lt, MIXED_BATCHES)[1])
    normalizer = pd.DataFrame([qrow("per-sample maximum (MINUT normalizer)", d118.grid_max_I.to_numpy()), qrow("TIC", d118.tic.to_numpy()), qrow("118.086 peak apex (0.82 min)", d118.main_apex.to_numpy()), qrow("118.086 level in [3,4) min (raw)", d118.bin3_max.to_numpy()),
                               qrow("118.086 level in [3,4) min / per-sample maximum", (d118.bin3_max / d118.grid_max_I).to_numpy())])
    normalizer.to_csv(OUT / "covid_normalizer.csv", index=False); display(normalizer)
    d118["normalizer_ion"] = d118.grid_max_mz.round(2).astype(str) + " @ " + d118.grid_max_rt.round(1).astype(str) + " min"; ions = d118.groupby(["normalizer_ion", "disease"]).size().unstack(fill_value=0).reset_index(); ions.to_csv(OUT / "covid_normalizer_ions.csv", index=False); display(ions)
    pd.DataFrame([dict(main_rt_mean=d118.main_rt.mean(), main_rt_min=d118.main_rt.min(), main_rt_max=d118.main_rt.max(), main_M1_over_M=d118.main_M1_over_M.mean(), expected=mn.iso_expect("C5H12NO2", 1), bin3_max_over_main_median=(d118.bin3_max / d118.main_apex).median(),
                       bin3_nonzero_min=d118.bin3_nonzero.min(), bin4_nonzero_min=d118.bin4_nonzero.min(), bin3_max_over_min_median=(d118.bin3_max / d118.bin3_min).median(), first_scan_I_median=d118.first_scan_I.median())]).to_csv(OUT / "covid_mz118_peak.csv", index=False)
    lr = normalizer.batches_sev_over_ctl[0]
    print(f"normalizer (per-sample maximum): severe/control within the mixed batches {lr}, van Elteren p = {normalizer.vanElteren_p[0]:.1e}; the normalizer ion is m/z 496.34 (LPC 16:0) in {int(d118.grid_max_mz.round(2).eq(496.34).sum())} of 63 runs; true 118.086 peak at {d118.main_rt.mean():.2f} min")
else:
    print("no raw COVID files and no per-sample table: sections 6-9 skipped")

19 s


,quantity,median_ctl,median_sev,batches_sev_over_ctl,vanElteren_p
0,per-sample maximum (MINUT normalizer),7.442616e+08,8.942409e+08,"[1.188, 1.415, 1.476, 1.211]",0.000273
1,TIC,6.603568e+11,6.643542e+11,"[0.971, 1.06, 0.848, 1.14]",0.122442
2,118.086 peak apex (0.82 min),7.824254e+07,7.032075e+07,"[0.781, 0.773, 0.788, 1.432]",0.554547
3,"118.086 level in [3,4) min (raw)",1.900924e+06,1.210536e+06,"[0.949, 1.012, 1.022, 1.024]",0.433645
4,"118.086 level in [3,4) min / per-sample maximum",2.234841e-03,1.527036e-03,"[0.822, 0.712, 0.644, 0.952]",0.000973


disease,normalizer_ion,Control,Severe
0,309.16 @ 11.6 min,1,0
1,353.27 @ 16.1 min,2,0
2,358.37 @ 14.5 min,0,1
3,358.37 @ 14.7 min,0,1
4,360.32 @ 17.3 min,1,3
5,360.32 @ 17.4 min,7,9
6,365.14 @ 13.0 min,4,0
7,381.3 @ 16.8 min,3,0
8,496.34 @ 15.7 min,3,0
9,496.34 @ 15.8 min,9,16


normalizer (per-sample maximum): severe/control within the mixed batches [1.188, 1.415, 1.476, 1.211], van Elteren p = 2.7e-04; the normalizer ion is m/z 496.34 (LPC 16:0) in 28 of 63 runs; true 118.086 peak at 0.82 min


## 7. The m/z 322.272 ion: three C₁₈H₃₇NO₂ isomers and the M+1 truncation (response letter)

The first version of the article annotated the window [322,322.5) × [15,16) min as sphingosine d18:1
[M+Na]⁺. The pass below extracts, between 13 and 17 min, the sodium adduct, the protonated ion, the
potassium adduct, the water-loss ions and the data-dependent MS/MS spectra of m/z 300.29 and 322.27
(fragments 282/264/252), and records the M+1 peak by intensity class of the monoisotopic peak.

In [9]:
from pyteomics import mzxml
M0 = 299.28243; SPH_NA, SPH_H = M0 + Na, M0 + H
ISO_IONS = {"MH": SPH_H, "MNa": SPH_NA, "MK": M0 + K, "MH_H2O": SPH_H - H2O, "MH_2H2O": SPH_H - 2 * H2O, "MNa_H2O": SPH_NA - H2O, "comp_322243": 322.2430}
ISOMERS = {"a_13.7": (13.45, 14.05, "MH"), "b_14.5": (14.2, 14.8, "MNa"), "c_15.8": (15.55, 16.1, "MNa")}

def process_322(path):
    name = Path(path).stem; rows, ms2 = [], []; win = {14844: (0.0, np.nan, np.nan), 16044: (0.0, np.nan, np.nan)}
    with mzxml.read(str(path), use_index=False) as r:
        for s in r:
            rt = float(s["retentionTime"])
            if rt < 13.0:
                continue
            if rt > 17.0:
                break
            mz = np.asarray(s["m/z array"], float); it = np.asarray(s["intensity array"], float)
            if s.get("msLevel") == 2:
                pmz = float(s["precursorMz"][0]["precursorMz"])
                if abs(pmz - SPH_H) <= 0.01 or abs(pmz - SPH_NA) <= 0.01:
                    ms2.append(dict(sample=name, rt=rt, precursor=pmz, target="300.29" if abs(pmz - SPH_H) <= 0.01 else "322.27", base=float(it.max()) if it.size else 0.0, f282=peak(mz, it, 282.27914), f264=peak(mz, it, 264.26858), f252=peak(mz, it, 252.26858)))
                continue
            if s.get("msLevel") != 1:
                continue
            w = (mz >= 300) & (mz <= 350)
            rows.append(dict(rt=rt, floor=float(it[w].min()) if w.any() else np.nan, M1=peak(mz, it, SPH_NA + D13C), M2=peak(mz, it, SPH_NA + 2 * D13C), half=peak(mz, it, SPH_NA + D13C / 2), **{k: peak(mz, it, v) for k, v in ISO_IONS.items()}))
            for b, (lo, hi) in {14844: (14, 15), 16044: (15, 16)}.items():
                if lo <= rt < hi:
                    i, m = bin_maximum(mz, it, 322.0, 322.5)
                    if i > win[b][0]:
                        win[b] = (i, m, rt)
    t = pd.DataFrame(rows); rec = dict(sample=name, disease=disease(name), batch=name[:8], floor_median=t.floor.median())
    for b, (i, m, rt) in win.items():
        rec[f"bin{b}_retained_mz"] = round(m, 4); rec[f"bin{b}_retained_rt"] = round(rt, 2); rec[f"bin{b}_retained_I"] = i
    for tag, (lo, hi, ref) in ISOMERS.items():
        w = t[(t.rt >= lo) & (t.rt < hi)]
        if w[ref].max() <= 0:
            continue
        a = w[ref].idxmax(); apex_rt = t.rt[a]; pk = t[(t.rt >= apex_rt - 0.1) & (t.rt <= apex_rt + 0.1)]; rec[f"{tag}_rt"] = round(apex_rt, 3); rec[f"{tag}_apex"] = pk[ref].max()
        if ref == "MNa":
            top = pk[pk.MNa >= 0.5 * pk.MNa.max()]; rec[f"{tag}_M1_over_M"] = top.M1.sum() / top.MNa.sum(); rec[f"{tag}_M2_over_M"] = top.M2.sum() / top.MNa.sum(); rec[f"{tag}_half_over_M"] = top.half.sum() / top.MNa.sum(); rec[f"{tag}_M1_missing"] = (top.M1 == 0).mean()
        for k in ISO_IONS:
            if k == ref:
                continue
            rec[f"{tag}_{k}_rel"] = pk[k].sum() / max(pk[ref].sum(), 1); rec[f"{tag}_{k}_corr"] = np.corrcoef(w[k], w[ref])[0, 1] if w[k].std() > 0 and w[ref].std() > 0 else np.nan
    return rec, t[t.MNa > 0][["rt", "MNa", "M1", "floor"]].assign(sample=name), pd.DataFrame(ms2)

per322 = OUT / "covid_mz322_per_sample.csv"; d322 = mn.cached(per322)
if d322 is None and HAVE_COVID_RAW and mn.raw_available("covid"):
    t0 = time.time(); res = Parallel(n_jobs=6)(delayed(process_322)(f) for f in files_covid())
    d322 = pd.DataFrame([r for r, _, _ in res]).sort_values("sample"); d322.to_csv(per322, index=False); pd.concat([s for _, s, _ in res]).to_csv(OUT / "covid_mz322_scans.csv", index=False); pd.concat([m for _, _, m in res if len(m)]).to_csv(OUT / "covid_ms2_300_322.csv", index=False)
    d322 = mn.cached(per322); print(f"{time.time() - t0:.0f} s")
if d322 is not None:
    S322 = pd.read_csv(OUT / "covid_mz322_scans.csv"); M2 = pd.read_csv(OUT / "covid_ms2_300_322.csv"); M2["target"] = M2.target.astype(float).round(2); sv = (d322.disease == "Severe").to_numpy(); lt = d322.batch.to_numpy(); rows = []
    for tag, (lo, hi, ref) in ISOMERS.items():
        row = dict(isomer=tag, reference_ion=ref, detected=d322[f"{tag}_rt"].notna().sum(), rt_mean=d322[f"{tag}_rt"].mean(), rt_sd=d322[f"{tag}_rt"].std(), apex_median=d322[f"{tag}_apex"].median(), sev_over_ctl_pooled=d322[sv][f"{tag}_apex"].median() / d322[~sv][f"{tag}_apex"].median(),
                   batches_sev_over_ctl=[round(v, 2) for v in batch_ratios(d322[f"{tag}_apex"], sv, lt, MIXED_BATCHES)], vanElteren_p=van_elteren(d322[f"{tag}_apex"].fillna(0), sv, lt, MIXED_BATCHES)[1])
        for k in ISO_IONS:
            if k != ref:
                row[f"{k}_rel"] = d322[f"{tag}_{k}_rel"].median(); row[f"{k}_corr"] = d322[f"{tag}_{k}_corr"].median()
        if ref == "MNa":
            row.update(M1_over_M=d322[f"{tag}_M1_over_M"].median(), M1_missing=d322[f"{tag}_M1_missing"].median(), M2_over_M=d322[f"{tag}_M2_over_M"].median(), half_over_M=d322[f"{tag}_half_over_M"].median())
        grp = M2[(M2.target == (300.29 if ref == "MH" else 322.27)) & (M2.rt >= lo) & (M2.rt < hi)]
        row.update(ms2_spectra=len(grp), ms2_injections=grp["sample"].nunique() if len(grp) else 0, ms2_f282=(grp.f282 > 0).mean() if len(grp) else np.nan, ms2_f264=(grp.f264 > 0).mean() if len(grp) else np.nan, ms2_f252=(grp.f252 > 0).mean() if len(grp) else np.nan); rows.append(row)
    isomers = pd.DataFrame(rows); isomers.to_csv(OUT / "covid_c18h37no2_isomers.csv", index=False); display(isomers.round(3))
    S322["cls"] = pd.cut(S322.MNa, [0, 5e4, 1e5, 2e5, 5e5, 1e6, 1e12], labels=["<50k", "50-100k", "100-200k", "200-500k", "0.5-1M", ">1M"])
    trunc = S322.groupby("cls", observed=True).agg(n_scans=("MNa", "size"), M1_sum=("M1", "sum"), M_sum=("MNa", "sum"), M1_missing=("M1", lambda s: (s == 0).mean()), floor=("floor", "median")); trunc["M1_over_M"] = trunc.M1_sum / trunc.M_sum; trunc["expected"] = mn.iso_expect("C18H37NO2Na", 1)
    trunc = trunc.reset_index(); trunc.to_csv(OUT / "covid_mz322_truncation.csv", index=False); display(trunc.round(3))
    a_ = isomers.set_index("isomer")

16 s


,isomer,reference_ion,detected,rt_mean,rt_sd,apex_median,sev_over_ctl_pooled,batches_sev_over_ctl,vanElteren_p,MNa_rel,MNa_corr,MK_rel,MK_corr,MH_H2O_rel,MH_H2O_corr,MH_2H2O_rel,MH_2H2O_corr,MNa_H2O_rel,MNa_H2O_corr,comp_322243_rel,comp_322243_corr,ms2_spectra,ms2_injections,ms2_f282,ms2_f264,ms2_f252,MH_rel,MH_corr,M1_over_M,M1_missing,M2_over_M,half_over_M
0,a_13.7,MH,63,13.703,0.095,2644856.000,1.377,"[0.6, 1.9, 1.35, 1.42]",0.073,0.414,0.996,0.000,0.566,0.241,0.993,0.0,-0.056,0.000,-0.048,0.000,-0.059,63,63,0.984,0.54,0.762,NaN,NaN,NaN,NaN,NaN,NaN
1,b_14.5,MNa,63,14.489,0.065,2508391.750,0.888,"[0.87, 1.04, 1.01, 2.06]",0.055,NaN,NaN,0.007,0.670,0.249,0.977,0.0,-0.073,0.000,-0.054,0.112,-0.101,96,58,0.000,0.00,0.000,0.070,0.560,0.187,0.0,0.007,0.0
2,c_15.8,MNa,63,15.826,0.040,627474.812,0.469,"[0.65, 1.9, 0.9, 1.33]",0.122,NaN,NaN,0.000,0.278,0.187,-0.095,0.0,-0.105,0.078,-0.074,0.089,-0.251,3,3,0.000,0.00,0.000,0.865,0.815,0.082,0.4,0.000,0.0


,cls,n_scans,M1_sum,M_sum,M1_missing,floor,M1_over_M,expected
0,<50k,4523,3.095450e+06,1.305972e+08,0.966,15770.460,0.024,0.198
1,50-100k,2618,6.159638e+06,1.916540e+08,0.885,14249.792,0.032,0.198
2,100-200k,1820,1.907191e+07,2.531181e+08,0.621,14546.986,0.075,0.198
3,200-500k,1171,5.118415e+07,3.617801e+08,0.210,13197.507,0.141,0.198
4,0.5-1M,570,7.144453e+07,4.085654e+08,0.044,13224.084,0.175,0.198
5,>1M,768,2.559814e+08,1.376823e+09,0.003,14126.280,0.186,0.198


## 8. Sensitivity of the ranking to the normalization and to the start of the RT grid (Limitations)

MINUT is re-run on the 63 files (MS$^1$ scans) with the grid of the article (2–20 min) and with a grid
starting at 0.5 min, each with the per-sample-maximum and the TIC normalization. For each of the four
settings we run the article pipeline in five-fold cross-validation and in leave-one-batch-out over the
four mixed batches, and record the Gini rank of fifteen reference ions and the 118.086 channel across all
RT windows.

In [10]:
def transform_variants(path):
    tic = 0.0; scans = []
    for rt, mz, it in mn.read_spectra(path, (1,)):
        tic += float(np.asarray(it, float).sum()); scans.append((rt, mz, it))
    return Path(path).stem, {"tic": tic, "A": mn.minut(scans, GRID), "B": mn.minut(scans, GRID_EXT)}

cache = OUT / "covid_minut_variants.npz"
if cache.exists():
    items = list(np.load(cache, allow_pickle=True)["items"])
elif mn.raw_available("covid"):
    t0 = time.time(); items = Parallel(n_jobs=6)(delayed(transform_variants)(f) for f in files_covid()); np.savez_compressed(cache, items=np.array(items, dtype=object)); print(f"MINUT re-run, two grids: {time.time() - t0:.0f} s")
else:
    items = None
HAVE_VARIANTS = items is not None
if HAVE_VARIANTS:
    runs_ix = subj.set_index("sample_name"); vnames = [n for n, _ in items]; vy = np.array([1 if runs_ix.loc[n, "disease"] == "Severe COVID-19" else 0 for n in vnames]); vbatches = np.array([n[:8] for n in vnames])
    V = {}
    for tag, grid in (("A", GRID), ("B", GRID_EXT)):
        for norm in ("max", "tic"):
            Xv = np.array([mn.feature_vector(*r[tag], grid, normalization=norm, tic=r["tic"]) for _, r in items])
            if norm == "tic":
                Xv[:, :Xv.shape[1] // 3] *= 1e3
            V[f"{tag}_{norm}"] = (Xv, grid, vy, vbatches)
    def rank_tools(Xk, yk):
        imp = RandomForestClassifier(n_estimators=1000, random_state=0, n_jobs=N_JOBS).fit(Xk, yk).feature_importances_; rk = np.empty(Xk.shape[1], int); rk[np.argsort(imp)[::-1]] = np.arange(1, Xk.shape[1] + 1)
        return imp, rk, list(np.argsort(imp)[::-1][:1000]), set(int(i) for i in mn.HybridSelector(100, n_jobs=N_JOBS).fit(Xk, yk).indices_)
    REF_IONS = {"propionylcarnitine (C3)": (218.1387, 0.86), "acetylcarnitine (C2)": (204.1230, 0.80), "octanoylcarnitine (C8)": (288.2169, 9.05), "decanoylcarnitine (C10)": (316.2482, 10.63), "palmitoylcarnitine (C16)": (400.3421, 14.22), "oleoylcarnitine (C18:1)": (426.3578, 14.43),
                "hypoxanthine": (137.0458, 0.88), "xanthine": (153.0407, 0.92), "tyrosine": (182.0812, 0.92), "kynurenic acid": (190.0499, 6.92), "sphingoid 300.290 (13.7 min)": (300.2897, 13.71), "LPC 16:0": (496.3398, 15.77), "cortisol": (363.2166, 9.52), "paracetamol-compatible 152.071": (152.0706, 4.05), "118.086 background [4,5)": (VAL, 4.5)}
    perf, mrows, chan = [], [], []
    for key, (Xv, grid, yv, lv) in V.items():
        tag, norm = key.split("_"); label = ("2-20 min" if tag == "A" else "0.5-20.5 min") + f", {norm}"; imp, rk, top1000, selected = rank_tools(Xv, yv); pipe = covid_pipeline(); acc, auc = [], []
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=2024).split(Xv, yv):
            f = pipe.fit(Xv[tr], yv[tr]); acc.append(accuracy_score(yv[te], f.predict(Xv[te]))); auc.append(roc_auc_score(yv[te], f.predict_proba(Xv[te])[:, 1]))
        mm = np.isin(lv, MIXED_BATCHES); Xmm, ymm, lmm = Xv[mm], yv[mm], lv[mm]; pr = np.zeros(len(ymm))
        for batch in MIXED_BATCHES:
            te = lmm == batch; f = pipe.fit(Xmm[~te], ymm[~te]); pr[te] = f.predict_proba(Xmm[te])[:, 1]
        perf.append(dict(setting=label, cv_accuracy=np.mean(acc), cv_accuracy_sd=np.std(acc), cv_auc=np.mean(auc), leave_one_batch_out_auc=roc_auc_score(ymm, pr), n_descriptors=Xv.shape[1]))
        nbv = Xv.shape[1] // 3; Mv = grid[0] + (grid[1] - grid[0]) * Xv[:, nbv:2 * nbv]; Rv = grid[3] + (grid[4] - grid[3]) * Xv[:, 2 * nbv:]; svv = yv == 1
        for mk, (mz, rt) in REF_IONS.items():
            b = bin_index(grid, mz, rt)
            if b is None:
                mrows.append(dict(setting=label, marker=mk, note="outside grid")); continue
            same = np.abs(Mv[:, b] - mz) <= mz * 10e-6; dv = Xv[:, b]
            mrows.append(dict(setting=label, marker=mk, bin=b, retained_share=same.mean(), gini_rank_I=int(rk[b]), n_descriptors=Xv.shape[1], hybrid_pos=(top1000.index(b) + 1) if b in top1000 else None, in_final_100=b in selected, pooled_sev_over_ctl=np.median(dv[svv]) / np.median(dv[~svv]) if np.median(dv[~svv]) > 0 else np.nan))
        rt = grid[3]
        while rt < grid[4] - 1e-9:
            b = bin_index(grid, VAL, rt + 0.01); same = np.abs(Mv[:, b] - VAL) <= VAL * 10e-6; dv = Xv[:, b]
            chan.append(dict(representation=f"re-run {label}", rt_interval=f"[{rt:g},{rt + grid[5]:g})", bin=b, retained_share=same.mean(), retained_rt=np.median(Rv[:, b]), I_median_ctl=np.median(dv[~svv]), I_median_sev=np.median(dv[svv]), sev_over_ctl=np.median(dv[svv]) / max(np.median(dv[~svv]), 1e-12),
                             batches_sev_over_ctl=[round(v, 2) for v in batch_ratios(dv, svv, lv, MIXED_BATCHES)], vanElteren_p=van_elteren(dv, svv, lv, MIXED_BATCHES)[1], gini_rank_I=int(rk[b]), hybrid_pos=(top1000.index(b) + 1) if b in top1000 else None, in_final_100=b in selected)); rt += grid[5]
        print(f"{label}: CV accuracy {np.mean(acc):.3f} ± {np.std(acc):.3f}, AUC {np.mean(auc):.3f}; leave-one-batch-out AUC {perf[-1]['leave_one_batch_out_auc']:.3f}", flush=True)
    sens = pd.DataFrame(perf); sens.to_csv(OUT / "covid_sensitivity_performance.csv", index=False); smark = pd.DataFrame(mrows); smark.to_csv(OUT / "covid_sensitivity_markers.csv", index=False); display(sens.round(3))
    pre = smark[smark.hybrid_pos.notna() & ~smark.marker.str.startswith(("118.086", "paracetamol", "LPC 16:0"))]
    print("reference ions among the 1 000 pre-selected descriptors (endogenous markers):", "none" if pre.empty else pre[["setting", "marker", "hybrid_pos"]].to_dict("records"))
else:
    print("no raw files: sections 8-9 skipped")

MINUT re-run, two grids: 24 s


2-20 min, max: CV accuracy 0.826 ± 0.029, AUC 0.904; leave-one-batch-out AUC 0.734


2-20 min, tic: CV accuracy 0.903 ± 0.082, AUC 0.960; leave-one-batch-out AUC 0.665


0.5-20.5 min, max: CV accuracy 0.856 ± 0.093, AUC 0.922; leave-one-batch-out AUC 0.699


0.5-20.5 min, tic: CV accuracy 0.888 ± 0.040, AUC 0.965; leave-one-batch-out AUC 0.783


,setting,cv_accuracy,cv_accuracy_sd,cv_auc,leave_one_batch_out_auc,n_descriptors
0,"2-20 min, max",0.826,0.029,0.904,0.734,64800
1,"2-20 min, tic",0.903,0.082,0.960,0.665,64800
2,"0.5-20.5 min, max",0.856,0.093,0.922,0.699,72000
3,"0.5-20.5 min, tic",0.888,0.040,0.965,0.783,72000


reference ions among the 1 000 pre-selected descriptors (endogenous markers): none


In [11]:
# the 118.086 channel in the deposited vectors, and the 100 descriptors the article's selector keeps on the whole cohort (within-batch test, MAF annotation, raw re-test of the peak-like retained maxima)
imp0, rank0, top1000_0, selected0 = rank_tools(X, y) if HAVE_VARIANTS else (None, None, None, None)
if HAVE_VARIANTS:
    rt = 2.0; rows = []
    while rt < 20 - 1e-9:
        b = bin_index(GRID, VAL, rt + 0.01); same = np.abs(M[:, b] - VAL) <= VAL * 10e-6; dv = X[:, b]
        rows.append(dict(representation="deposited vectors (2-20 min, max)", rt_interval=f"[{rt:g},{rt + 1:g})", bin=b, retained_share=same.mean(), retained_rt=np.median(R[:, b]), I_median_ctl=np.median(dv[~sev]), I_median_sev=np.median(dv[sev]), sev_over_ctl=np.median(dv[sev]) / max(np.median(dv[~sev]), 1e-12),
                         batches_sev_over_ctl=[round(v, 2) for v in batch_ratios(dv, sev, batches, MIXED_BATCHES)], vanElteren_p=van_elteren(dv, sev, batches, MIXED_BATCHES)[1], gini_rank_I=int(rank0[b]), hybrid_pos=(top1000_0.index(b) + 1) if b in top1000_0 else None, in_final_100=b in selected0)); rt += 1
    channel = pd.DataFrame(rows + chan); channel.to_csv(OUT / "covid_mz118_channel_bins.csv", index=False); display(channel[channel.representation.str.startswith("deposited")].round(4))
    dep_ch = channel[channel.representation.str.startswith("deposited")]
    print(f"118.086 channel (deposited vectors): lower in severe in {int((dep_ch.sev_over_ctl < 1).sum())} of {len(dep_ch)} RT bins, van Elteren p <= 0.007 in {int((dep_ch.vanElteren_p <= 0.007).sum())} of them")
    maf_c = None
    if mn.raw_available("maf"):
        maf_c = pd.read_csv(next((mn.RAW / "maf").glob("m_MTBLS2291*maf.tsv")), sep="\t", dtype=str, usecols=["chemical_formula", "metabolite_identification", "retention_time"]).dropna(subset=["chemical_formula"])
        maf_c["M"] = maf_c.chemical_formula.map(lambda f: mn.formula_mass(f) if isinstance(f, str) else np.nan); maf_c = maf_c.dropna(subset=["M"]); maf_c["rt"] = pd.to_numeric(maf_c.retention_time, errors="coerce")
    sel_full = mn.HybridSelector(100, n_jobs=N_JOBS).fit(X, y); rows = []
    for pos, idx in enumerate(sel_full.indices_, 1):
        blk, b = divmod(int(idx), NB); mz_lo, rt_lo = 100 + 0.5 * (b % 1200), 2 + (b // 1200); mode = float(pd.Series(M[:, b].round(3)).value_counts().index[0]); same = np.abs(M[:, b] - mode) <= mode * 5e-6; dv = X[:, int(idx)]; hits = []
        if maf_c is not None:
            for a, dm in (("[M+H]+", H), ("[M+Na]+", Na), ("[M+NH4]+", NH4), ("[M+K]+", K)):
                ok = maf_c[np.abs(maf_c.M + dm - mode) <= mode * 5e-6]; hits += [f"{r_.metabolite_identification} ({r_.chemical_formula}, {a}, MAF RT {r_.rt})" for r_ in ok.itertuples()]
        rows.append(dict(position=pos, descriptor=int(idx), block=["I", "mz", "RT"][blk], bin=b, mz_bin=f"[{mz_lo:g},{mz_lo + 0.5:g})", rt_interval=f"[{rt_lo:g},{rt_lo + 1:g})", retained_mz=mode, retained_share=same.mean(), retained_rt=np.median(R[same, b]), retained_rt_sd=np.std(R[same, b]),
                         pooled_sev_over_ctl=np.median(dv[sev]) / np.median(dv[~sev]) if np.median(dv[~sev]) > 0 else np.nan, batches_sev_over_ctl=[round(v, 2) for v in batch_ratios(dv, sev, batches, MIXED_BATCHES)], vanElteren_p=van_elteren(dv, sev, batches, MIXED_BATCHES)[1], gini_rank=int(rank0[int(idx)]), maf_match="; ".join(hits[:3])))
    top100 = pd.DataFrame(rows); top100.to_csv(OUT / "covid_top100_descriptors.csv", index=False); display(top100.head(12))
    n_int = int((top100.block == "I").sum()); n_lower = int(((top100.block == "I") & (top100.pooled_sev_over_ctl < 1)).sum())
    print(f"{n_int} of the 100 selected descriptors are intensity descriptors; {n_lower} of them are lower in severe cases (pooled)")

,representation,rt_interval,bin,retained_share,retained_rt,I_median_ctl,I_median_sev,sev_over_ctl,batches_sev_over_ctl,vanElteren_p,gini_rank_I,hybrid_pos,in_final_100
0,"deposited vectors (2-20 min, max)","[2,3)",36,1.0000,2.1416,0.0023,0.0017,0.7397,"[0.81, 0.75, 0.71, 0.78]",0.0005,47,47.0,True
1,"deposited vectors (2-20 min, max)","[3,4)",1236,1.0000,3.2524,0.0022,0.0015,0.6968,"[0.81, 0.67, 0.67, 0.95]",0.0008,39,39.0,True
2,"deposited vectors (2-20 min, max)","[4,5)",2436,1.0000,4.5449,0.0021,0.0015,0.6961,"[0.81, 0.73, 0.69, 0.73]",0.0003,101,101.0,True
3,"deposited vectors (2-20 min, max)","[5,6)",3636,0.7143,5.8976,0.0023,0.0016,0.7178,"[0.77, 0.72, 0.75, 0.74]",0.0010,24190,NaN,False
4,"deposited vectors (2-20 min, max)","[6,7)",4836,0.3175,6.1160,0.0026,0.0020,0.7681,"[1.05, 0.71, 0.93, 1.02]",0.0599,19470,NaN,False
5,"deposited vectors (2-20 min, max)","[7,8)",6036,1.0000,7.2273,0.0019,0.0015,0.7942,"[0.9, 0.75, 0.74, 0.75]",0.0067,779,779.0,False
6,"deposited vectors (2-20 min, max)","[8,9)",7236,0.9524,8.2162,0.0017,0.0014,0.8317,"[0.81, 0.78, 0.87, 0.74]",0.0049,19823,NaN,False
7,"deposited vectors (2-20 min, max)","[9,10)",8436,0.9524,9.1660,0.0016,0.0013,0.8200,"[0.83, 0.77, 0.8, 0.79]",0.0045,30647,NaN,False
8,"deposited vectors (2-20 min, max)","[10,11)",9636,1.0000,10.2919,0.0015,0.0012,0.7942,"[0.88, 0.73, 0.64, 0.68]",0.0003,29307,NaN,False
9,"deposited vectors (2-20 min, max)","[11,12)",10836,0.9841,11.3209,0.0014,0.0011,0.8051,"[0.75, 0.7, 0.67, 0.7]",0.0003,32256,NaN,False


118.086 channel (deposited vectors): lower in severe in 17 of 18 RT bins, van Elteren p <= 0.007 in 17 of them


,position,descriptor,block,bin,mz_bin,rt_interval,retained_mz,retained_share,retained_rt,retained_rt_sd,pooled_sev_over_ctl,batches_sev_over_ctl,vanElteren_p,gini_rank,maf_match
0,1,58316,RT,15116,"[458,458.5)","[14,15)",458.311,0.444444,14.167900,0.012942,0.937362,"[1.0, 0.96, 1.0, 1.0]",0.012632,349,
1,2,52726,RT,9526,"[663,663.5)","[9,10)",663.349,0.301587,9.538317,0.017749,0.936864,"[0.93, 0.95, 0.95, 0.95]",0.002398,379,
2,3,10858,I,10858,"[129,129.5)","[11,12)",129.030,0.746032,11.894950,0.148069,0.541718,"[0.85, 0.73, 1.37, 0.52]",0.013173,204,"Barbituric acid (C4H4N2O3, [M+H]+, MAF RT 21.378); 5-hydroxyuracil..."
3,4,2436,I,2436,"[118,118.5)","[4,5)",118.086,1.000000,4.544917,0.300977,0.696072,"[0.81, 0.73, 0.69, 0.73]",0.000299,101,"Valine (C5H11NO2, [M+H]+, MAF RT 1.245); Valine (C5H11NO2, [M+H]+,..."
4,5,1236,I,1236,"[118,118.5)","[3,4)",118.086,1.000000,3.252450,0.235716,0.696825,"[0.81, 0.67, 0.67, 0.95]",0.000810,39,"Valine (C5H11NO2, [M+H]+, MAF RT 1.245); Valine (C5H11NO2, [M+H]+,..."
5,6,18030,I,18030,"[115,115.5)","[17,18)",115.014,0.444444,17.937833,0.089581,0.519304,"[0.87, 0.55, 0.99, 0.7]",0.013173,403,
6,7,2648,I,2648,"[224,224.5)","[4,5)",224.128,1.000000,4.874333,0.299928,0.660229,"[0.86, 0.69, 0.74, 0.73]",0.000224,34,"rac-Anhalonidine (C12H17NO3, [M+H]+, MAF RT 1.24); DL-Cerulenin (C..."
7,8,9658,I,9658,"[129,129.5)","[10,11)",129.030,0.698413,10.891633,0.227581,0.548193,"[0.82, 0.68, 1.34, 0.56]",0.013173,864,"Barbituric acid (C4H4N2O3, [M+H]+, MAF RT 21.378); 5-hydroxyuracil..."
8,9,13258,I,13258,"[129,129.5)","[13,14)",129.030,0.809524,13.918633,0.243110,0.506875,"[0.82, 0.72, 1.48, 0.48]",0.022952,120,"Barbituric acid (C4H4N2O3, [M+H]+, MAF RT 21.378); 5-hydroxyuracil..."
9,10,12394,I,12394,"[297,297.5)","[12,13)",297.222,0.730159,12.234025,0.022373,0.657444,"[0.77, 0.74, 0.7, 1.0]",0.002859,69,


83 of the 100 selected descriptors are intensity descriptors; 83 of them are lower in severe cases (pooled)


## 9. Literature markers of COVID-19 severity (Table S24) and their identities

The candidate list (`inputs/covid_literature_candidates_*.csv`, 212 compounds) is expanded to the
positive-mode adducts and every ion is extracted (±5 ppm, 0.3–20.5 min) from the 63 files. We call a
candidate ion *usable* when it gives a consistent chromatographic peak in at least 32 injections (apex RT
IQR ≤ 0.3 min, apex ≥ 3 × the median scan). Three further conditions define the rows of Table S24:

* *sourced*: the compound is named, with its direction, in the literature table deposited with the code
  (`inputs/covid_literature_review.md`, 21 studies; class-wide statements such as "all acylcarnitines" do
  not count), or belongs to the open-search list, whose rows each cite their primary studies
  (`inputs/covid_literature_candidates_open.csv`). The preliminary list also holds compounds we added as isobaric or alternative hypotheses (the
  N-acylethanolamines, for instance) and acylcarnitines coded "up" without a source; they stay in the
  computation but not in the table;
* *chemically consistent*: the ion is the protonated molecule, or it co-elutes (±0.2 min) with another
  usable adduct of the same compound. An adduct whose apex lies minutes away from the compound's [M+H]⁺
  is an isobaric ion of unknown identity;
* *discriminative within the batches*: the ion is the bin maximum in at least half of the runs
  (otherwise the intensity descriptor measures another ion) and its raw apex intensity differs between
  severe cases and controls within the four mixed batches (van Elteren |z| ≥ 2.5). Rank and z always
  belong to the same ion.

For the ranking we use one method, the one of the second lung configuration: the intensity descriptor of
the ion's bin in a 1 000-tree forest fitted on the 63 subjects, with the normalization of the article
(per-sample maximum) and all bins. The rank is given as median [min–max] over ten forest seeds, because
with 63 subjects the single-seed rank of a descriptor varies by an order of magnitude. The grid is
extended to 0.5–20.5 min because the polar markers (amino acids, purines, polyamines, short-chain
acylcarnitines) elute in the void volume, before the 2-min start of the grid of the main text; the
2–20 min rank is reported alongside when the ion lies in it. We do not rank the m/z and RT descriptors
here: in this batch-confounded cohort they encode the mass calibration and the retention drift of the
acquisition days rather than the chemistry.

In [12]:
MASS = {"C": 12.0, "H": 1.00782503207, "N": 14.0030740048, "O": 15.99491461956, "P": 30.97376163, "S": 31.972071, "Na": 22.9897692809, "K": 38.96370668, "Cl": 34.96885268, "F": 18.99840322}; E_ = 0.00054857990946
ADDUCTS = {"[M+H]+": (MASS["H"] - E_, 1), "[M+Na]+": (MASS["Na"] - E_, 1), "[M+K]+": (MASS["K"] - E_, 1), "[M+NH4]+": (MASS["N"] + 4 * MASS["H"] - E_, 1), "[M+H-H2O]+": (MASS["H"] - E_ - 2 * MASS["H"] - MASS["O"], 1), "[M+2H]2+": (2 * (MASS["H"] - E_), 2)}
import re as _re
def neutral_mass(f):
    return sum(MASS[el] * int(n or 1) for el, n in _re.findall(r"([A-Z][a-z]?)(\d*)", f))

def lit_targets():
    C = pd.concat([pd.read_csv(f) for f in sorted(INPUTS.glob("covid_literature_candidates*.csv"))]).drop_duplicates("name"); C = C[C.formula.notna() & (C.formula.astype(str).str.len() > 1)].copy(); C["M"] = C.formula.map(lambda f: neutral_mass(str(f).rstrip("+"))); rows = []
    for r_ in C.itertuples():
        if str(r_.formula).endswith("+"):
            rows.append(dict(name=r_.name, formula=r_.formula, adduct="[M]+", mz=neutral_mass(str(r_.formula).rstrip("+")) - E_, z=1)); continue
        for ad, (dm, z) in ADDUCTS.items():
            if z == 2 and r_.M < 600:
                continue
            mz = (r_.M + dm) / z
            if ad == "[M+H-H2O]+" and not any(k in r_.name.lower() for k in ("acid", "ol", "sphing", "cer ", "lpc", "lpe", "glycer", "cortis", "sterone", "inosine", "ose")):
                continue
            if 100 <= mz < 700:
                rows.append(dict(name=r_.name, formula=r_.formula, adduct=ad, mz=mz, z=z))
    T = pd.DataFrame(rows); T["cls"] = T.name.map(dict(zip(C.name, C["class"]))); T["direction"] = T.name.map(dict(zip(C.name, C.direction_severe.fillna("")))); return T, C

def process_lit(args):
    path, tg_mz = args; tg_mz = np.asarray(tg_mz); lo, hi = tg_mz * (1 - 5e-6), tg_mz * (1 + 5e-6); lo1, hi1 = (tg_mz + 1.00336) * (1 - 5e-6), (tg_mz + 1.00336) * (1 + 5e-6); rts, XX, X1 = [], [], []
    for rt, mz, it in read_scans(path, 0.3, 20.5, 1):
        if np.any(np.diff(mz) < 0):
            o = np.argsort(mz); mz, it = mz[o], it[o]
        a, b = np.searchsorted(mz, lo), np.searchsorted(mz, hi); a1, b1 = np.searchsorted(mz, lo1), np.searchsorted(mz, hi1)
        XX.append(np.array([it[i:j].max() if j > i else 0.0 for i, j in zip(a, b)])); X1.append(np.array([it[i:j].max() if j > i else 0.0 for i, j in zip(a1, b1)])); rts.append(rt)
    rts = np.array(rts); XX = np.array(XX); X1 = np.array(X1); out = []
    for k in range(len(tg_mz)):
        x = XX[:, k]
        if x.max() <= 0:
            out.append(dict(sample=Path(path).stem, k=k, detected=False)); continue
        j = int(np.argmax(x)); half = x >= 0.5 * x[j]; l = j
        while l > 0 and half[l - 1]: l -= 1
        r_ = j
        while r_ < len(x) - 1 and half[r_ + 1]: r_ += 1
        out.append(dict(sample=Path(path).stem, k=k, detected=True, apex_rt=float(rts[j]), I_apex=float(x[j]), fwhm_min=float(rts[r_] - rts[l]), M1_over_M=float(X1[j, k] / x[j]), I_median_scan=float(np.median(x)), frac_scans_present=float(np.mean(x > 0))))
    return out

if HAVE_VARIANTS:
    T, CAND = lit_targets(); print(f"{len(T)} target ions from {T.name.nunique()} candidates ({len(CAND)} compounds in the list)")
    perlit = OUT / "covid_lit_xic_per_sample.csv"; PL = pd.read_csv(perlit) if perlit.exists() else None
    if PL is None and mn.raw_available("covid"):
        t0 = time.time(); res = Parallel(n_jobs=6)(delayed(process_lit)((f, T.mz.to_numpy())) for f in files_covid()); PL = pd.DataFrame([r_ for rows_ in res for r_ in rows_]).merge(T.reset_index().rename(columns={"index": "k"}), on="k"); PL.to_csv(perlit, index=False); print(f"XIC pass: {time.time() - t0:.0f} s")
    HAVE_LIT = PL is not None
else:
    HAVE_LIT = False

910 target ions from 211 candidates (211 compounds in the list)


XIC pass: 67 s


In [13]:
# provenance: which candidates are named, with a verified direction, in the literature table (preliminary list) or come from the open-search list (sources in its CSV)
import re
SUB_ = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
def norm_name(s_):
    s_ = re.sub(r"\(.*?\)", "", str(s_)).lower().replace("-", " ").replace("_", " "); return re.sub(r"\s+", " ", s_).strip()
def name_match(cand, rev):
    b = norm_name(rev)
    for a in (norm_name(part) for part in str(cand).split("/")):
        if (a == b) if len(a) < 5 else (a == b or re.search(r"(?<![a-z])" + re.escape(a) + r"(?![a-z])", b) is not None):
            return True
    return False
rev_txt = (INPUTS / "covid_literature_review.md").read_text(encoding="utf-8"); rev_txt = rev_txt[rev_txt.index("## Master table"):]
REV = []
for line in rev_txt.splitlines():
    if not line.startswith("| ") or line.startswith("| Metabolite") or line.startswith("|---"):
        continue
    c_ = [x.strip() for x in line.strip().strip("|").split("|")]
    if len(c_) >= 7 and not any(k in c_[0].lower() for k in ("class", "all 18", "unnamed feature", "further significant")):
        REV.append(dict(review_name=re.sub(r"\*+", "", c_[0]), direction=re.sub(r"\*+", "", c_[1]), formula=re.sub(r"[^A-Za-z0-9]", "", c_[2].translate(SUB_)), consistency=re.sub(r"\*+", "", c_[6])))
REV = pd.DataFrame(REV)
prov = []
for f_ in sorted(INPUTS.glob("covid_literature_candidates*.csv")):
    C_ = pd.read_csv(f_); is_open = "open" in f_.name
    for r_ in C_.itertuples():
        if is_open:
            prov.append(dict(name=r_.name, formula=r_.formula, cls=r_[3], list="open", sourced=True, source=f"open-search list: {str(r_.notes)[:160]}")); continue
        f = re.sub(r"[^A-Za-z0-9]", "", str(r_.formula))
        hit = [x for x in REV.itertuples() if name_match(r_.name, x.review_name) and (not re.fullmatch(r"C\d+H\d+[A-Za-z0-9]*", x.formula) or x.formula == f) and "not found" not in x.consistency.lower() and "unverified" not in x.consistency.lower()]
        prov.append(dict(name=r_.name, formula=r_.formula, cls=r_[3], list="prelim", sourced=bool(hit), source="; ".join(f"{x.review_name} [{x.direction}; {x.consistency}]" for x in hit[:3])))
PROV = pd.DataFrame(prov).drop_duplicates("name"); PROV.to_csv(OUT / "covid_literature_provenance.csv", index=False)
SOURCED = set(PROV[PROV.sourced].name)
print(f"{len(PROV)} candidates; {len(SOURCED)} sourced in the reviews ({int((PROV.list == 'open').sum())} open-search list, {int(((PROV.list == 'prelim') & PROV.sourced).sum())} preliminary list); {int((~PROV.sourced).sum())} not sourced, of which {int(((~PROV.sourced) & (PROV.cls != 'drug')).sum())} endogenous")

212 candidates; 127 sourced in the reviews (82 open-search list, 45 preliminary list); 85 not sourced, of which 60 endogenous

In [14]:
def load_variant(tag):
    Ii = np.array([r_[tag][0].ravel() for _, r_ in items]); MZ = np.array([r_[tag][1].ravel() for _, r_ in items]); RT = np.array([r_[tag][2].ravel() for _, r_ in items])
    return dict(I=Ii, MZ=MZ, RT=RT, y=vy, batches=vbatches, grid=GRID if tag == "A" else GRID_EXT)

def intensity_ranks(X, yk, seeds=range(10)):
    out = []
    for sd in seeds:
        imp = RandomForestClassifier(n_estimators=1000, random_state=sd, n_jobs=N_JOBS).fit(X, yk).feature_importances_; rk = np.empty(len(imp), int); rk[np.argsort(imp)[::-1]] = np.arange(1, len(imp) + 1); out.append(rk)
    return np.array(out)

if HAVE_LIT:
    PL["sev"] = PL["sample"].map(lambda n: runs_ix.loc[n, "disease"] == "Severe COVID-19"); PL["batch"] = PL["sample"].str[:8]; rows = []
    for (name, ad), g in PL.groupby(["name", "adduct"]):
        dd = g[g.detected == True]  # noqa: E712
        if len(dd) < 32:
            rows.append(dict(name=name, adduct=ad, mz=g.mz.iloc[0], detected=len(dd), usable=False)); continue
        rt_iqr = dd.apex_rt.quantile(0.75) - dd.apex_rt.quantile(0.25); peaky = (dd.I_apex / dd.I_median_scan.clip(lower=1)).median(); z, p = van_elteren(dd.I_apex.to_numpy(), dd.sev.to_numpy(), dd.batch.to_numpy(), MIXED_BATCHES); ratios = batch_ratios(dd.I_apex.to_numpy(), dd.sev.to_numpy(), dd.batch.to_numpy(), MIXED_BATCHES)
        rows.append(dict(name=name, adduct=ad, mz=round(g.mz.iloc[0], 4), detected=len(dd), usable=bool(rt_iqr <= 0.3 and peaky >= 3), apex_rt=round(dd.apex_rt.median(), 2), rt_iqr=round(rt_iqr, 2), I_apex=round(dd.I_apex.median()), apex_over_median=round(peaky, 1), M1_over_M=round(dd.M1_over_M.median(), 3), within_batch_z=round(z, 2), within_batch_p=f"{p:.1e}", batch_ratios=[round(v, 2) for v in ratios]))
    A = pd.DataFrame(rows); A.to_csv(OUT / "covid_lit_xic_summary.csv", index=False); use = A[A.usable == True].copy()  # noqa: E712
    use["cls"] = use.name.map(dict(zip(CAND.name, CAND["class"]))); use["sourced"] = use.name.isin(SOURCED)
    def consistent_(r_):
        if r_.adduct == "[M+H]+":
            return True
        o_ = use[(use.name == r_["name"]) & (use.adduct != r_.adduct)]; return bool(len(o_) and (o_.apex_rt - r_.apex_rt).abs().min() <= 0.2)
    use["consistent"] = use.apply(consistent_, axis=1)
    print(f"{len(use)} usable ions ({use.name.nunique()} compounds) give a consistent chromatographic peak in >= 32 injections; {int(use.sourced.sum())} ions of {use[use.sourced].name.nunique()} sourced compounds")
    t0 = time.time(); rows, tops = [], []
    for tag, glabel in (("B", "0.5-20.5"), ("A", "2-20")):
        D = load_variant(tag); grid = D["grid"]; X = D["I"] / D["I"].max(1, keepdims=True); RK = intensity_ranks(X, D["y"]); med = np.median(RK, axis=0)
        occ_mode = np.array([np.nanmedian(D["MZ"][:, b]) if np.isfinite(D["MZ"][:, b]).any() else np.nan for b in range(X.shape[1])])
        if tag == "B":   # the hundred most important intensity descriptors of the chosen configuration (median rank over seeds), annotated with literature masses
            for pos, b in enumerate(np.argsort(med)[:100], 1):
                w = occ_mode[b]; hits = use[np.abs(use.mz - w) <= w * 10e-6] if np.isfinite(w) else use.iloc[0:0]
                sh = float(np.nanmean(np.abs(D["MZ"][:, b] - w) <= w * 10e-6)) if np.isfinite(w) else 0.0
                tops.append(dict(position=pos, bin=int(b), bin_limits=mn.bin_label(int(b), grid), retained_mz=round(float(w), 4) if np.isfinite(w) else None, retained_share=round(sh, 2), rank_median=float(med[b]), rank_min=int(RK[:, b].min()), rank_max=int(RK[:, b].max()),
                                 sev_over_ctl=float(np.median(X[D["y"] == 1, b]) / max(np.median(X[D["y"] == 0, b]), 1e-12)), literature_match="; ".join(f"{h.name} {h.adduct} (z {h.within_batch_z:+.1f}, share {np.nanmean(np.abs(D['MZ'][:, b] - h.mz) <= h.mz * 10e-6):.2f})" for h in hits.itertuples())))
        for r_ in use.itertuples():
            b = bin_index(grid, r_.mz, r_.apex_rt)
            if b is None:
                rows.append(dict(grid=glabel, name=r_.name, adduct=r_.adduct, mz=r_.mz, apex_rt=r_.apex_rt, cls=r_.cls, sourced=r_.sourced, consistent=r_.consistent, in_grid=False)); continue
            v = RK[:, b]
            rows.append(dict(grid=glabel, name=r_.name, adduct=r_.adduct, mz=r_.mz, apex_rt=r_.apex_rt, cls=r_.cls, sourced=r_.sourced, consistent=r_.consistent, in_grid=True, bin=int(b), bin_limits=mn.bin_label(int(b), grid), retained_share=round(float(np.nanmean(np.abs(D["MZ"][:, b] - r_.mz) <= r_.mz * 10e-6)), 2),
                             within_batch_z=r_.within_batch_z, batch_ratios=r_.batch_ratios, n_descriptors=X.shape[1], rank_median=float(np.median(v)), rank_min=int(v.min()), rank_max=int(v.max()), seeds_top100=int((v <= 100).sum())))
        print(f"grid {glabel}: {X.shape[1]} intensity descriptors, ten forests ({time.time() - t0:.0f} s)")
    LR = pd.DataFrame(rows); LR.to_csv(OUT / "covid_literature_marker_ranks.csv", index=False); TP = pd.DataFrame(tops); TP.to_csv(OUT / "covid_literature_top100.csv", index=False)

423 usable ions (181 compounds) give a consistent chromatographic peak in >= 32 injections; 258 ions of 111 sourced compounds


grid 0.5-20.5: 24000 intensity descriptors, ten forests (37 s)


grid 2-20: 21600 intensity descriptors, ten forests (74 s)


In [15]:
if HAVE_LIT:
    rule = (LR.in_grid == True) & (LR.cls != "drug") & (LR.retained_share >= 0.5) & (LR.within_batch_z.abs() >= 2.5)  # noqa: E712
    S24 = LR[rule & LR.sourced & LR.consistent & (LR.grid == "0.5-20.5")].sort_values("rank_median").copy()
    r220 = LR[(LR.grid == "2-20") & (LR.in_grid == True)].set_index(["name", "adduct"])  # noqa: E712
    S24["rank_2_20"] = [f"{int(r220.loc[(n, a), 'rank_median'])} [{int(r220.loc[(n, a), 'rank_min'])}--{int(r220.loc[(n, a), 'rank_max'])}]" if (n, a) in r220.index else "--" for n, a in zip(S24.name, S24.adduct)]
    for c_ in ("rank_min", "rank_max", "seeds_top100", "n_descriptors"):
        S24[c_] = S24[c_].astype(int)
    S24["identity"] = np.where(S24.name.isin(["tyrosine", "propionylcarnitine (C3)", "acetylcarnitine (C2)", "decanoylcarnitine (C10)", "octanoylcarnitine (C8)", "palmitoylcarnitine (C16)", "myristoylcarnitine (C14)", "stearoylcarnitine (C18)", "hypoxanthine", "kynurenic acid"]), "MS/MS", "mass")
    S24["direction_lit"] = S24.name.map(dict(zip(PROV.name, PROV.source))).str.replace(r"open-search list: ", "", regex=True).str.split("|").str[0].str.strip().str.slice(0, 60)
    excluded = LR[rule & LR.sourced & ~LR.consistent & (LR.grid == "0.5-20.5")]
    print(f"Table S24: {len(S24)} ions of {S24.name.nunique()} sourced compounds (grid 0.5-20.5 min, per-sample maximum, all bins, {int(S24.n_descriptors.iloc[0])} intensity descriptors); {len(excluded)} sourced discriminative ions excluded as orphan adducts: " + ", ".join(f"{r_.name} {r_.adduct} ({r_.apex_rt:.2f} min)" for r_ in excluded.itertuples()))
    display(S24[["name", "adduct", "apex_rt", "retained_share", "within_batch_z", "identity", "rank_median", "rank_min", "rank_max", "seeds_top100", "rank_2_20", "direction_lit"]])
    best = S24.iloc[0]; best220 = LR[rule & LR.sourced & LR.consistent & (LR.grid == "2-20")].sort_values("rank_median").iloc[0]
    print(f"best-ranked sourced marker, 0.5-20.5 min: {best['name']} {best.adduct} at {int(best.rank_median)} [{int(best.rank_min)}-{int(best.rank_max)}] of {int(best.n_descriptors)}; 2-20 min: {best220['name']} {best220.adduct} at {int(best220.rank_median)} [{int(best220.rank_min)}-{int(best220.rank_max)}] of {int(best220.n_descriptors)}")
    ent = TP[TP.literature_match.fillna("") != ""]; print("top-100 intensity descriptors (median rank) whose retained maximum matches a literature mass:"); display(ent[["position", "bin", "retained_mz", "retained_share", "rank_median", "sev_over_ctl", "literature_match"]])
    # LaTeX fragment of Table S24 (blue, marked-manuscript convention)
    def tex_(s_): return str(s_).replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")
    lines = [r"\begin{table}[H]\centering\small", r"\captionsetup{font={color=blue}}",
             rf"\caption{{COVID-19: Gini rank of the literature severity markers that are named with a verified direction in the deposited literature table or open-search list, whose ion is the maximum of its MINUT bin in at least half of the 63 runs and differs between severe patients and controls within the four mixed acquisition batches (stratified Wilcoxon $|z| \geq 2.5$ on the raw apex intensity; positive: higher in severe), and whose adduct is the protonated molecule or co-elutes with another adduct of the compound. Rank of the bin's intensity descriptor in a 1\,000-tree forest fitted on the 63 subjects with the normalization of the main text (per-sample maximum) on the grid extended to the void volume (0.5--20.5~min, {int(best.n_descriptors)} bins): median [min--max] over ten forest seeds; the last column gives the same rank on the 2--20~min grid of the main text (--: the ion elutes before 2~min). Identity: MS/MS = diagnostic fragments present in the data-dependent MS/MS spectra acquired on the ion; mass = exact mass and retention time only. No marker ion is among the 100 selected descriptors in any of the forests.}}",
             r"\label{si:tab:covid_batch_markers}", r"\begin{adjustbox}{max width=\textwidth}", r"\begin{tabular}{@{}llrrrlll@{}}\toprule",
             r"Marker & Ion & RT (min) & Retained-maximum share & Within-batch $z$ & Identity & Rank, 0.5--20.5 min & Rank, 2--20 min \\\midrule"]
    for r_ in S24.itertuples():
        lines.append(f"{tex_(r_.name)} & {tex_(r_.adduct)} & {r_.apex_rt:.2f} & {r_.retained_share:.2f} & ${r_.within_batch_z:+.1f}$ & {r_.identity} & {int(r_.rank_median)} [{int(r_.rank_min)}--{int(r_.rank_max)}] & {r_.rank_2_20} " + r"\\")
    lines += [r"\bottomrule\end{tabular}", r"\end{adjustbox}", r"\end{table}"]
    (OUT / "tex").mkdir(exist_ok=True); (OUT / "tex" / "tableS24_literature_markers.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
    NUM = dict(n_candidates=len(PROV), n_sourced=len(SOURCED), n_usable_compounds=int(use.name.nunique()), n_usable_sourced_compounds=int(use[use.sourced].name.nunique()), n_table_ions=len(S24), n_table_compounds=int(S24.name.nunique()), n_excluded_orphan=len(excluded),
               n_descriptors_ext=int(best.n_descriptors), n_descriptors_220=int(best220.n_descriptors), best_ext=dict(name=best['name'], adduct=best.adduct, median=int(best.rank_median), min=int(best.rank_min), max=int(best.rank_max), seeds_top100=int(best.seeds_top100)),
               best_220=dict(name=best220['name'], adduct=best220.adduct, median=int(best220.rank_median), min=int(best220.rank_min), max=int(best220.rank_max)), any_marker_top100_all_seeds=bool((S24.seeds_top100 == 10).any()))
    json.dump(NUM, open(OUT / "covid_literature_numbers.json", "w"), indent=1)

Table S24: 11 ions of 11 sourced compounds (grid 0.5-20.5 min, per-sample maximum, all bins, 24000 intensity descriptors); 9 sourced discriminative ions excluded as orphan adducts: 13-HODE [M+Na]+ (14.32 min), 3-Hydroxyoctanoylcarnitine [M+NH4]+ (14.74 min), 9-HODE [M+Na]+ (14.32 min), Homovanillic acid [M+H-H2O]+ (0.93 min), Methionine sulfoxide [M+NH4]+ (10.84 min), N-Acetylmethionine [M+Na]+ (6.38 min), Suberoylcarnitine (C8-DC) [M+NH4]+ (14.48 min), acetylcarnitine (C2) [M+Na]+ (8.10 min), sphingosine (d18:1) [M+H-H2O]+ (16.11 min)


,name,adduct,apex_rt,retained_share,within_batch_z,identity,rank_median,rank_min,rank_max,seeds_top100,rank_2_20,direction_lit
152,N1-Acetylspermidine,[M+H]+,0.77,0.57,2.69,mass,223.5,77,490,1,--,up severe & death; paradoxical down baseline predicts worse
10,3-Hydroxyisovalerylcarnitine (C5-OH),[M+H]+,1.02,0.54,4.17,mass,931.0,119,9283,0,--,up COVID+ vs -; up up severe
415,tyrosine,[M+H]+,0.93,1.00,3.41,MS/MS,2934.5,919,9226,0,--,Tyrosine [Conflicting; Conflicting]
323,octanoylcarnitine (C8),[M+H]+,9.06,1.00,2.78,MS/MS,4829.0,3057,7645,0,4990 [1619--6992],Octanoylcarnitine (C8) [Conflicting; Conflicting]
369,propionylcarnitine (C3),[M+H]+,0.86,1.00,3.59,MS/MS,6192.5,210,9399,0,--,Propionylcarnitine (C3) [Conflicting; Conflicting]
368,proline,[M+Na]+,0.79,0.52,2.68,mass,9235.0,3442,9914,0,--,Proline [↓ mostly; Consistent]
193,acetylcarnitine (C2),[M+H]+,0.81,1.00,3.04,MS/MS,9271.0,532,10037,0,--,Acetylcarnitine (C2) [No change; Null]
303,linoleic acid,[M+Na]+,16.23,1.00,2.83,mass,14570.0,3254,15534,0,13759 [2942--13813],Linoleic acid (18:2) [↑; Consistent]
202,arachidonic acid,[M+Na]+,16.17,1.00,2.69,mass,15467.0,1316,15513,0,12105 [12081--13739],Arachidonic acid (20:4) [Conflicting (axis-dependent); Confl
236,decanoylcarnitine (C10),[M+H]+,10.64,1.00,2.56,MS/MS,20568.0,236,20631,0,18215 [18198--18279],Decanoylcarnitine (C10) [↓ (severity-progression only); Diff


best-ranked sourced marker, 0.5-20.5 min: N1-Acetylspermidine [M+H]+ at 223 [77-490] of 24000; 2-20 min: octanoylcarnitine (C8) [M+H]+ at 4990 [1619-6992] of 21600
top-100 intensity descriptors (median rank) whose retained maximum matches a literature mass:


,position,bin,retained_mz,retained_share,rank_median,sev_over_ctl,literature_match
50,51,17162,281.2465,0.56,90.5,2.540923,"linoleic acid [M+H]+ (z +1.5, share 0.56)"
74,75,16220,410.2663,0.78,128.0,2.733047,"myristoylcarnitine (C14) [M+K]+ (z +0.1, share 0.78)"
93,94,14606,203.0523,0.67,166.0,3.475522,"Mannose [M+Na]+ (z +0.7, share 0.67)"


Identity of the markers of Table S24 in the raw data: MS$^1$ apex at the expected retention time, isotope
ratios, co-eluting adducts and in-source fragments, and the data-dependent MS/MS spectra acquired on the
ion (precursor within 0.01 Da, inside the RT window), scored for the diagnostic fragments of each compound.

In [16]:
ID_TARGETS = {
    "tyrosine [M+H]+": dict(mz=182.08117, rt=0.92, formula="C9H12NO3", comp={"[M+Na]+ 204.063": 182.08117 - H + Na, "-NH3 165.055": 165.05462, "-HCOOH 136.076": 136.07569, "123.044": 123.04406}, frag={"165.055": 165.05462, "136.076": 136.07569, "123.044": 123.04406, "119.049": 119.04914, "91.054": 91.05423}),
    "propionylcarnitine C3 [M+H]+": dict(mz=218.13869, rt=0.86, formula="C10H20NO4", comp={"[M+Na]+ 240.121": 218.13869 - H + Na, "-N(CH3)3 159.065": 159.06519, "85.028": 85.02841}, frag={"85.028": 85.02841, "60.081": 60.08078, "159.065": 159.06519, "57.034 (propionyl)": 57.03349}),
    "acetylcarnitine C2 [M+H]+": dict(mz=204.12304, rt=0.80, formula="C9H18NO4", comp={"[M+Na]+ 226.105": 204.12304 - H + Na, "-N(CH3)3 145.050": 145.04954, "85.028": 85.02841}, frag={"85.028": 85.02841, "60.081": 60.08078, "145.050": 145.04954, "43.018 (acetyl)": 43.01784}),
    "decanoylcarnitine C10 [M+H]+": dict(mz=316.24824, rt=10.63, formula="C17H34NO4", comp={"[M+Na]+ 338.230": 316.24824 - H + Na, "-N(CH3)3 257.175": 257.17474, "85.028": 85.02841}, frag={"85.028": 85.02841, "60.081": 60.08078, "257.175": 257.17474, "155.143 (decanoyl)": 155.14304}),
    "octanoylcarnitine C8 [M+H]+": dict(mz=288.21694, rt=9.05, formula="C15H30NO4", comp={"[M+Na]+ 310.199": 288.21694 - H + Na, "-N(CH3)3 229.143": 229.14344, "85.028": 85.02841}, frag={"85.028": 85.02841, "60.081": 60.08078, "229.143": 229.14344, "127.112 (octanoyl)": 127.11174}),
    "palmitoylcarnitine C16 [M+H]+": dict(mz=400.34214, rt=14.22, formula="C23H46NO4", comp={"[M+Na]+ 422.324": 400.34214 - H + Na, "-N(CH3)3 341.269": 341.26864, "85.028": 85.02841}, frag={"85.028": 85.02841, "60.081": 60.08078, "341.269": 341.26864, "239.237 (palmitoyl)": 239.23694}),
    "myristoylcarnitine C14 [M+H]+": dict(mz=372.31083, rt=13.33, formula="C21H42NO4", comp={"[M+Na]+ 394.293": 372.31083 - H + Na, "-N(CH3)3 313.237": 313.23733, "85.028": 85.02841}, frag={"85.028": 85.02841, "60.081": 60.08078, "313.237": 313.23733, "211.206 (myristoyl)": 211.20564}),
    "stearoylcarnitine C18 [M+H]+": dict(mz=428.37343, rt=14.93, formula="C25H50NO4", comp={"[M+Na]+ 450.356": 428.37343 - H + Na, "-N(CH3)3 369.300": 369.29993, "85.028": 85.02841}, frag={"85.028": 85.02841, "60.081": 60.08078, "369.300": 369.29993, "267.268 (stearoyl)": 267.26824}),
    "hypoxanthine [M+H]+": dict(mz=137.04579, rt=0.89, formula="C5H5N4O", comp={"[M+Na]+ 159.028": 137.04579 - H + Na, "-H2O 119.035": 119.03523, "-HCN 110.035": 110.03489}, frag={"119.035": 119.03523, "110.035": 110.03489, "94.040": 94.03997, "82.040": 82.03997, "67.029": 67.02908, "55.029": 55.02908}),
    "kynurenic acid [M+H]+": dict(mz=190.04986, rt=6.93, formula="C10H8NO3", comp={"[M+Na]+ 212.032": 190.04986 - H + Na, "-H2O 172.039": 172.03930, "-H2O-CO 144.044": 144.04439}, frag={"172.039": 172.03930, "144.044": 144.04439, "116.049": 116.04948, "89.039": 89.03858}),
    "cortisol [M+H]+": dict(mz=363.21660, rt=9.52, formula="C21H31O5", comp={"[M+Na]+ 385.199": 363.21660 - H + Na, "-H2O 345.206": 345.20604, "-2H2O 327.196": 327.19547}, frag={"345.206": 345.20604, "327.196": 327.19547, "309.185": 309.18491, "121.065": 121.06479, "97.065": 97.06479}),
    "paracetamol-compatible 152.071": dict(mz=152.07060, rt=4.05, formula="C8H10NO2", comp={"[M+Na]+ 174.053": 152.07060 - H + Na, "110.060": 110.06004}, frag={"110.060": 110.06004, "93.034": 93.03349, "65.039": 65.03858}),
}

def process_identity_covid(path):
    name = Path(path).stem; rows = []; ms2 = []
    with mzxml.read(str(path), use_index=False) as r:
        for s in r:
            rt = float(s["retentionTime"]); mz = np.asarray(s["m/z array"], float); it = np.asarray(s["intensity array"], float)
            if s.get("msLevel") == 2:
                pmz = float(s["precursorMz"][0]["precursorMz"])
                for tname, t_ in ID_TARGETS.items():
                    if abs(pmz - t_["mz"]) <= 0.01 and abs(rt - t_["rt"]) <= 0.5:
                        order = np.argsort(it)[::-1][:6]
                        ms2.append(dict(sample=name, target=tname, rt=round(rt, 3), precursor=round(pmz, 4), base=float(it.max()) if it.size else 0.0, **{f"frag {k}": peak(mz, it, v, 10e-6) for k, v in t_["frag"].items()}, top=";".join(f"{mz[j]:.4f}:{it[j]:.0f}" for j in order)))
                continue
            if s.get("msLevel") != 1:
                continue
            for tname, t_ in ID_TARGETS.items():
                if abs(rt - t_["rt"]) <= 0.6:
                    rows.append(dict(target=tname, rt=rt, M=peak(mz, it, t_["mz"]), M1=peak(mz, it, t_["mz"] + D13C), M2=peak(mz, it, t_["mz"] + 2 * D13C), **{k: peak(mz, it, v) for k, v in t_["comp"].items()}))
    Tt = pd.DataFrame(rows); out = []
    for tname, t_ in ID_TARGETS.items():
        g = Tt[Tt.target == tname]
        if g.empty or g.M.max() <= 0:
            out.append(dict(sample=name, target=tname, apex_I=0.0)); continue
        a = g.M.idxmax(); apex_rt = g.rt[a]; pk = g[(g.rt >= apex_rt - 0.08) & (g.rt <= apex_rt + 0.08)]; top = pk[pk.M >= 0.5 * pk.M.max()]
        rec = dict(sample=name, target=tname, apex_rt=round(apex_rt, 3), apex_I=pk.M.max(), M1_over_M=top.M1.sum() / top.M.sum(), M2_over_M=top.M2.sum() / top.M.sum())
        for k in t_["comp"]:
            rec[f"{k} rel"] = pk[k].sum() / max(pk.M.sum(), 1); rec[f"{k} corr"] = np.corrcoef(g.M, g[k])[0, 1] if g[k].std() > 0 and g.M.std() > 0 else np.nan
        out.append(rec)
    return pd.DataFrame(out), pd.DataFrame(ms2)

per_idc = OUT / "covid_marker_identity_per_sample.csv"
if not per_idc.exists() and HAVE_COVID_RAW and mn.raw_available("covid"):
    t0 = time.time(); res = Parallel(n_jobs=6)(delayed(process_identity_covid)(f) for f in files_covid()); Dc = pd.concat([a for a, _ in res]); M2c = pd.concat([b for _, b in res if len(b)])
    Dc.to_csv(per_idc, index=False); M2c.to_csv(OUT / "covid_marker_identity_ms2.csv", index=False); print(f"{time.time() - t0:.0f} s")
if per_idc.exists():
    Dc = pd.read_csv(per_idc); M2c = pd.read_csv(OUT / "covid_marker_identity_ms2.csv"); rows = []
    for tname, t_ in ID_TARGETS.items():
        g = Dc[Dc.target == tname]; det = g[g.apex_I > 0]; m_ = M2c[M2c.target == tname]
        rec = dict(target=tname, mz=t_["mz"], detected=f"{len(det)}/{len(g)}", apex_rt=det.apex_rt.median(), apex_rt_sd=det.apex_rt.std(), apex_I_median=det.apex_I.median(), M1_over_M=det.M1_over_M.median(), M1_expected=mn.iso_expect(t_["formula"], 1), M2_over_M=det.M2_over_M.median(), M2_expected=mn.iso_expect(t_["formula"], 2), ms2_spectra=len(m_), ms2_injections=m_["sample"].nunique() if len(m_) else 0)
        for k in t_["comp"]:
            rec[f"{k} rel"] = det[f"{k} rel"].median(); rec[f"{k} corr"] = det[f"{k} corr"].median()
        for k in t_["frag"]:
            rec[f"MS2 {k}"] = (m_[f"frag {k}"] > 0).mean() if len(m_) else np.nan
        rows.append(rec)
    idc = pd.DataFrame(rows); idc.to_csv(OUT / "covid_marker_identity.csv", index=False); display(idc[["target", "detected", "apex_rt", "M1_over_M", "M1_expected", "ms2_spectra", "ms2_injections"] + [c for c in idc.columns if c.startswith("MS2 85.028") or c.startswith("MS2 165") or c.startswith("MS2 144")]].round(3))
    ic = idc.set_index("target")

29 s


,target,detected,apex_rt,M1_over_M,M1_expected,ms2_spectra,ms2_injections,MS2 165.055,MS2 85.028,MS2 144.044
0,tyrosine [M+H]+,63/63,0.927,0.093,0.101,301,63,1.0,NaN,NaN
1,propionylcarnitine C3 [M+H]+,63/63,0.863,0.099,0.112,84,48,NaN,1.0,NaN
2,acetylcarnitine C2 [M+H]+,63/63,0.808,0.099,0.101,70,63,NaN,1.0,NaN
3,decanoylcarnitine C10 [M+H]+,63/63,10.641,0.184,0.188,74,63,NaN,1.0,NaN
4,octanoylcarnitine C8 [M+H]+,63/63,9.061,0.159,0.166,69,63,NaN,1.0,NaN
5,palmitoylcarnitine C16 [M+H]+,63/63,14.232,0.242,0.252,73,63,NaN,1.0,NaN
6,myristoylcarnitine C14 [M+H]+,63/63,13.334,0.222,0.231,60,60,NaN,1.0,NaN
7,stearoylcarnitine C18 [M+H]+,63/63,14.931,0.264,0.274,56,56,NaN,1.0,NaN
8,hypoxanthine [M+H]+,62/63,0.895,0.008,0.069,9,6,NaN,NaN,NaN
9,kynurenic acid [M+H]+,63/63,6.932,0.087,0.112,45,42,NaN,NaN,1.0


## 10. Two further checks of the Supporting Information

The valine/betaine peak (C₅H₁₂NO₂⁺, 0.82 min) lies outside the 2–20 min grid of the article. Grids
starting between 0.00 and 0.75 min (two of them with 0.5-min RT bins) bring it into a bin, whose retained maximum
and rank we then compute under the four configurations. The second check applies the second lung
configuration (stability filter on the 63 subjects, TIC proxy and PQN) to the COVID-19 cohort, in
cross-validation (filter and PQN reference recomputed in each training fold) and leave-one-batch-out.

In [17]:
# helpers of the second lung configuration applied to the COVID-19 cohort (stability filter, PQN, full Gini / hybrid ranking)
def stability_c(MZ, rows, thr=0.9, ppm=10e-6):
    Mr = MZ[rows]; ok = ~np.isnan(Mr); occ = ok.mean(0); mode = np.full(Mr.shape[1], np.nan); share = np.zeros(Mr.shape[1])
    for b in np.flatnonzero(occ >= thr):
        col = Mr[:, b]; md_ = np.nanmedian(col); mode[b] = md_; share[b] = np.mean(np.abs(col - md_) <= md_ * ppm)
    return (occ >= thr) & (share >= thr), mode, share

def pqn_c(S, ref_rows):
    ref = np.median(S[ref_rows], axis=0); ok = ref > 0; q = np.array([np.median(row[ok & (row > 0)] / ref[ok & (row > 0)]) if (ok & (row > 0)).any() else 1.0 for row in S]); return S / q[:, None]

def rank_all(Xk, yk):
    imp = RandomForestClassifier(n_estimators=1000, random_state=0, n_jobs=N_JOBS).fit(Xk, yk).feature_importances_; rg = np.empty(len(imp), int); rg[np.argsort(imp)[::-1]] = np.arange(1, len(imp) + 1)
    hyb = mn.HybridSelector(k=Xk.shape[1], pre_rank=Xk.shape[1], n_jobs=N_JOBS).fit(Xk, yk).indices_; rh = np.empty(Xk.shape[1], int); rh[hyb] = np.arange(1, Xk.shape[1] + 1)
    r_ = np.apply_along_axis(rankdata, 0, Xk); n1 = yk.sum(); n0 = len(yk) - n1; return rg, rh, (r_[yk == 1].sum(0) - n1 * (n1 + 1) / 2) / (n1 * n0), imp

VGRIDS = {"start 0.00 min": (100.0, 700.0, 0.5, 0.0, 20.0, 1.0), "start 0.25 min": (100.0, 700.0, 0.5, 0.25, 20.25, 1.0), "start 0.50 min": (100.0, 700.0, 0.5, 0.5, 20.5, 1.0), "start 0.75 min": (100.0, 700.0, 0.5, 0.75, 20.75, 1.0),
          "start 0.50 min, 0.5-min RT bins": (100.0, 700.0, 0.5, 0.5, 20.5, 0.5), "start 0.60 min, 0.5-min RT bins": (100.0, 700.0, 0.5, 0.6, 20.6, 0.5)}
VAL_MZ, VAL_RT = 118.0863, 0.82
def transform_vgrids(path):
    scans = [(rt, np.asarray(mz, float), np.asarray(it, float)) for rt, mz, it in mn.read_spectra(path, (1,))]
    return Path(path).stem, {tag: mn.minut(scans, grid) for tag, grid in VGRIDS.items()}

vg_file = OUT / "covid_valine_grid_ranks.csv"
if not vg_file.exists() and HAVE_VARIANTS and mn.raw_available("covid"):
    t0 = time.time(); vitems = Parallel(n_jobs=6)(delayed(transform_vgrids)(f) for f in files_covid()); vn = [n for n, _ in vitems]; assert vn == vnames; rows = []
    for tag, grid in VGRIDS.items():
        Iv = np.array([r_[tag][0].ravel() for _, r_ in vitems]); MZv = np.array([r_[tag][1].ravel() for _, r_ in vitems]); RTv = np.array([r_[tag][2].ravel() for _, r_ in vitems]); nbv = Iv.shape[1]
        b = bin_index(grid, VAL_MZ, VAL_RT); share = float(np.nanmean(np.abs(MZv[:, b] - VAL_MZ) <= VAL_MZ * 10e-6)); rt_med = float(np.nanmedian(RTv[:, b])); allrows = np.arange(len(vy)); svv = vy == 1
        mask, mode, st_share = stability_c(MZv, allrows); S = Iv / Iv.sum(1, keepdims=True) * 1e3; PQ = pqn_c(S, allrows); MX = Iv / Iv.max(1, keepdims=True); zraw, praw = van_elteren(Iv[:, b], svv, vbatches, MIXED_BATCHES)
        for cname, (J, cols_) in {"article (max, all bins)": (MX, np.arange(nbv)), "PQN, all bins": (PQ, np.arange(nbv)), "filter + max": (MX, np.flatnonzero(mask)), "filter + PQN": (PQ, np.flatnonzero(mask))}.items():
            cmc = {int(c): i for i, c in enumerate(cols_)}
            if b not in cmc:
                rows.append(dict(grid=tag, config=cname, bin=int(b), retained_share=round(share, 2), retained_rt=round(rt_med, 2), n_descriptors=len(cols_), note="bin not stable (excluded by the filter)")); continue
            Xk = J[:, cols_]; j = cmc[b]; rg, rh, auc_, imp = rank_all(Xk, vy); z, p = van_elteren(Xk[:, j], svv, vbatches, MIXED_BATCHES)
            rows.append(dict(grid=tag, config=cname, bin=int(b), retained_share=round(share, 2), retained_rt=round(rt_med, 2), n_descriptors=len(cols_), gini_rank=int(rg[j]), hybrid_rank=int(rh[j]), auc_pooled=round(float(auc_[j]), 3), within_batch_z=round(float(z), 2), within_batch_p=f"{p:.2g}", within_batch_z_raw=round(float(zraw), 2), batch_ratios=[round(v, 2) for v in batch_ratios(Xk[:, j], svv, vbatches, MIXED_BATCHES)]))
    vg = pd.DataFrame(rows); vg.to_csv(vg_file, index=False); print(f"{time.time() - t0:.0f} s")
if vg_file.exists():
    vg = pd.read_csv(vg_file); ok_ = vg[vg.gini_rank.notna()]; display(vg.drop(columns=["batch_ratios"]).round(3))

605 s


,grid,config,bin,retained_share,retained_rt,n_descriptors,gini_rank,hybrid_rank,auc_pooled,within_batch_z,within_batch_p,within_batch_z_raw
0,start 0.00 min,"article (max, all bins)",36,1.0,0.81,24000,6565,9294,0.547,0.44,0.66,-0.59
1,start 0.00 min,"PQN, all bins",36,1.0,0.81,24000,9126,4499,0.342,-0.76,0.45,-0.59
2,start 0.00 min,filter + max,36,1.0,0.81,6177,3842,3203,0.547,0.44,0.66,-0.59
3,start 0.00 min,filter + PQN,36,1.0,0.81,6177,3514,2035,0.342,-0.76,0.45,-0.59
4,start 0.25 min,"article (max, all bins)",36,1.0,0.81,24000,9261,9570,0.547,0.44,0.66,-0.59
5,start 0.25 min,"PQN, all bins",36,1.0,0.81,24000,9098,4610,0.348,-0.76,0.45,-0.59
6,start 0.25 min,filter + max,36,1.0,0.81,6187,1666,3150,0.547,0.44,0.66,-0.59
7,start 0.25 min,filter + PQN,36,1.0,0.81,6187,655,2087,0.348,-0.76,0.45,-0.59
8,start 0.50 min,"article (max, all bins)",36,1.0,0.81,24000,9875,9089,0.547,0.44,0.66,-0.59
9,start 0.50 min,"PQN, all bins",36,1.0,0.81,24000,9015,4651,0.344,-0.69,0.49,-0.59


In [18]:
if HAVE_VARIANTS:
    def config2_c(D, rows_, how="pqn", thr=0.9):
        mask, mode, share = stability_c(D["MZ"], rows_, thr); S = D["I"] / D["I"].sum(1, keepdims=True) * 1e3; J = pqn_c(S, rows_) if how == "pqn" else (S if how == "summax" else D["I"] / D["I"].max(1, keepdims=True)); cols_ = np.flatnonzero(mask)
        return J[:, cols_], cols_
    rows_perf = []
    for tag in ("A", "B"):
        D = load_variant(tag); yv_ = D["y"]; lv_ = D["batches"]
        for how in ("max", "pqn"):
            acc, auc = [], []
            for tr, te in StratifiedKFold(5, shuffle=True, random_state=2024).split(D["I"], yv_):
                Xk, cols_ = config2_c(D, tr, how); kp = mn.HybridSelector(100, n_jobs=N_JOBS).fit(Xk[tr], yv_[tr]).indices_; f = RandomForestClassifier(n_estimators=100, random_state=0).fit(Xk[tr][:, kp], yv_[tr])
                acc.append(accuracy_score(yv_[te], f.predict(Xk[te][:, kp]))); auc.append(roc_auc_score(yv_[te], f.predict_proba(Xk[te][:, kp])[:, 1]))
            mm = np.isin(lv_, MIXED_BATCHES); idx = np.flatnonzero(mm); pr = np.zeros(len(idx)); yt = yv_[idx]
            for batch in MIXED_BATCHES:
                te = lv_[idx] == batch; tr_rows = idx[~te]; Xk, cols_ = config2_c(D, tr_rows, how); kp = mn.HybridSelector(100, n_jobs=N_JOBS).fit(Xk[tr_rows], yv_[tr_rows]).indices_
                pr[te] = RandomForestClassifier(n_estimators=100, random_state=0).fit(Xk[tr_rows][:, kp], yv_[tr_rows]).predict_proba(Xk[idx[te]][:, kp])[:, 1]
            Xk, cols_ = config2_c(D, np.arange(len(yv_)), how)
            rows_perf.append(dict(grid="2-20" if tag == "A" else "0.5-20.5", normalization=how, threshold=0.9, n_stable=len(cols_), cv5_acc=round(np.mean(acc), 3), cv5_sd=round(np.std(acc), 3), cv5_auc=round(np.mean(auc), 3), lolo_acc=round(accuracy_score(yt, pr >= 0.5), 3), lolo_auc=round(roc_auc_score(yt, pr), 3))); print(rows_perf[-1], flush=True)
    config2_perf = pd.DataFrame(rows_perf); config2_perf.to_csv(OUT / "covid_stable_performance.csv", index=False); display(config2_perf)
    s2 = config2_perf[(config2_perf.grid == "2-20") & (config2_perf.normalization == "pqn")].iloc[0]

{'grid': '2-20', 'normalization': 'max', 'threshold': 0.9, 'n_stable': 5536, 'cv5_acc': 0.794, 'cv5_sd': 0.084, 'cv5_auc': 0.873, 'lolo_acc': 0.63, 'lolo_auc': 0.66}


{'grid': '2-20', 'normalization': 'pqn', 'threshold': 0.9, 'n_stable': 5536, 'cv5_acc': 0.903, 'cv5_sd': 0.082, 'cv5_auc': 0.945, 'lolo_acc': 0.565, 'lolo_auc': 0.589}


{'grid': '0.5-20.5', 'normalization': 'max', 'threshold': 0.9, 'n_stable': 6264, 'cv5_acc': 0.713, 'cv5_sd': 0.083, 'cv5_auc': 0.846, 'lolo_acc': 0.457, 'lolo_auc': 0.408}


{'grid': '0.5-20.5', 'normalization': 'pqn', 'threshold': 0.9, 'n_stable': 6264, 'cv5_acc': 0.872, 'cv5_sd': 0.096, 'cv5_auc': 0.921, 'lolo_acc': 0.457, 'lolo_auc': 0.54}


,grid,normalization,threshold,n_stable,cv5_acc,cv5_sd,cv5_auc,lolo_acc,lolo_auc
0,2-20,max,0.9,5536,0.794,0.084,0.873,0.630,0.660
1,2-20,pqn,0.9,5536,0.903,0.082,0.945,0.565,0.589
2,0.5-20.5,max,0.9,6264,0.713,0.083,0.846,0.457,0.408
3,0.5-20.5,pqn,0.9,6264,0.872,0.096,0.921,0.457,0.540
